# Benchmark de convolução 2D no Colab
Use **Ambiente de execução → Executar tudo** (Ctrl+F9). As células abaixo gravam os arquivos do projeto (código C legível) e a **última célula** compila, roda e baixa o CSV.
Para mudar o que roda (imagens, repetições, kernels), edite o bloco de configuração da última célula.

In [3]:
!mkdir -p /content/Artigo_PCD-main/src /content/Artigo_PCD-main/results


In [4]:
%%writefile /content/Artigo_PCD-main/Makefile
# Makefile - 2D convolution benchmark (OpenMP + Pthreads)
#
# Build strategy (matches the project conventions):
#   - Fixed optimization across the whole matrix: -O3 -march=native
#   - The SEQUENTIAL reference object is compiled WITHOUT -fopenmp so the
#     reference path is a genuine serial build. The convolution translation
#     unit guards its OpenMP code with #ifdef _OPENMP, so it builds both ways;
#     we compile it once WITH OpenMP for the harness (the harness needs the
#     parallel entry points) and additionally produce a seq-only object to
#     prove the reference compiles cleanly without -fopenmp.
#   - Parallel paths use -fopenmp -pthread.
#
# Targets:
#   make            build the conv_bench harness
#   make check      build + run the quick smoke matrix (fast, tiny images)
#   make run        build + run the full default matrix
#   make seqcheck   prove the sequential TU compiles WITHOUT -fopenmp
#   make selftest   build + run the standalone correctness smoke driver
#   make clean      remove build artifacts

CC      ?= gcc
CSTD    := -std=c11
WARN    := -Wall -Wextra
OPT     := -O3 -march=native
OMP     := -fopenmp -pthread
LDLIBS  := -lm

BUILD   := build
SRC     := src
BIN     := $(BUILD)/conv_bench
SELFTEST_BIN := $(BUILD)/selftest

SRCS    := $(SRC)/kernels.c $(SRC)/image.c $(SRC)/convolution.c \
           $(SRC)/energy.c $(SRC)/metrics.c $(SRC)/platform.c $(SRC)/main.c

# The selftest driver reuses the core engine TUs (no main.c, no energy/metrics)
# and provides its own main() guarded behind -DSELFTEST. platform.c is needed
# because convolution.c routes its thread calls through the platform layer.
SELFTEST_SRCS := $(SRC)/kernels.c $(SRC)/image.c $(SRC)/convolution.c \
                 $(SRC)/platform.c $(SRC)/selftest.c

.PHONY: all check run seqcheck selftest clean

all: $(BIN)

$(BIN): $(SRCS) | $(BUILD)
	$(CC) $(CSTD) $(WARN) $(OPT) $(OMP) $(SRCS) -o $(BIN) $(LDLIBS)

$(BUILD):
	mkdir -p $(BUILD)

# Quick smoke matrix: tiny images, few reps, all patterns/models, threads 1..8.
check: $(BIN)
	./$(BIN) --quick

# Full default matrix (3 patterns x {3,7} x {2k,4k,8k} x all models x 1,2,4,8).
run: $(BIN)
	./$(BIN)

# Prove the sequential reference compiles WITHOUT -fopenmp (serial build).
seqcheck: | $(BUILD)
	$(CC) $(CSTD) $(WARN) $(OPT) -c $(SRC)/convolution.c -o $(BUILD)/convolution_seq.o
	@echo "OK: convolution.c compiled without -fopenmp"

# Standalone correctness smoke driver (src/selftest.c, guarded by -DSELFTEST).
# Builds and runs the pixel-by-pixel agreement + PGM/PPM round-trip checks.
$(SELFTEST_BIN): $(SELFTEST_SRCS) | $(BUILD)
	$(CC) $(CSTD) $(WARN) $(OPT) $(OMP) -DSELFTEST $(SELFTEST_SRCS) -o $(SELFTEST_BIN) $(LDLIBS)

selftest: $(SELFTEST_BIN)
	./$(SELFTEST_BIN)

clean:
	rm -rf $(BUILD)


Writing /content/Artigo_PCD-main/Makefile


In [5]:
%%writefile /content/Artigo_PCD-main/src/convolution.c
/*
 * convolution.c - Four convolution implementations + validation.
 *
 * All four share convolve_pixel() so the arithmetic (and thus the output) is
 * byte-identical regardless of how work is partitioned. Border policy is
 * zero-padding (see convolution.h).
 *
 * The OpenMP paths are guarded by _OPENMP so this translation unit still
 * compiles when built WITHOUT -fopenmp (the sequential reference object is
 * built that way). When compiled without OpenMP, the OpenMP-named entry points
 * fall back to the sequential computation so the API stays complete.
 */
#include "convolution.h"
#include "platform.h"

#include <math.h>
#include <stdlib.h>

#ifdef _OPENMP
#include <omp.h>
#endif

/*
 * Compute a single output pixel (one channel) at (ox, oy, c) using the kernel.
 * Centered window, zero-padding outside the image. Shared by every variant.
 */
static inline float convolve_pixel(const image_t *in, const kernel_t *kern,
                                   int ox, int oy, int c)
{
    const int k = kern->k;
    const int r = k / 2;
    const int w = in->width;
    const int h = in->height;
    const int ch = in->channels;
    const float *src = in->data;
    const float *kd = kern->data;

    float acc = 0.0f;
    for (int ky = 0; ky < k; ++ky) {
        int iy = oy + ky - r;
        if (iy < 0 || iy >= h)
            continue; /* zero-padding: whole row contributes 0 */
        for (int kx = 0; kx < k; ++kx) {
            int ix = ox + kx - r;
            if (ix < 0 || ix >= w)
                continue; /* zero-padding */
            float s = src[((size_t)iy * w + ix) * ch + c];
            acc += s * kd[ky * k + kx];
        }
    }
    return acc;
}

/* Convolve a contiguous band of output rows [y0, y1). */
static void convolve_rows_range(const image_t *in, const kernel_t *kern,
                                image_t *out, int y0, int y1)
{
    const int w = in->width;
    const int ch = in->channels;
    for (int oy = y0; oy < y1; ++oy) {
        for (int ox = 0; ox < w; ++ox) {
            for (int c = 0; c < ch; ++c) {
                out->data[((size_t)oy * w + ox) * ch + c] =
                    convolve_pixel(in, kern, ox, oy, c);
            }
        }
    }
}

void convolve_sequential(const image_t *in, const kernel_t *kern,
                         image_t *out)
{
    convolve_rows_range(in, kern, out, 0, in->height);
}

/* ---------------------------------------------------------------- Pthreads */

typedef struct {
    const image_t *in;
    const kernel_t *kern;
    image_t *out;
    int y0, y1;
} pthr_arg_t;

static void *pthr_worker(void *p)
{
    pthr_arg_t *a = (pthr_arg_t *)p;
    convolve_rows_range(a->in, a->kern, a->out, a->y0, a->y1);
    return NULL;
}

void convolve_pthreads(const image_t *in, const kernel_t *kern,
                       image_t *out, int nthreads)
{
    const int h = in->height;
    if (nthreads < 1)
        nthreads = 1;
    if (nthreads > h)
        nthreads = h > 0 ? h : 1;

    if (nthreads == 1) {
        convolve_rows_range(in, kern, out, 0, h);
        return;
    }

    plat_thread_t *tids = (plat_thread_t *)malloc((size_t)nthreads * sizeof(plat_thread_t));
    pthr_arg_t *args = (pthr_arg_t *)malloc((size_t)nthreads * sizeof(pthr_arg_t));
    if (!tids || !args) {
        /* Allocation failed: fall back to a correct sequential pass. */
        free(tids);
        free(args);
        convolve_rows_range(in, kern, out, 0, h);
        return;
    }

    /* Track which threads were actually spawned. A separate boolean array is
     * used (instead of a sentinel value in tids[]) so the sentinel does not
     * depend on the concrete representation of plat_thread_t across
     * platforms. */
    char *joinable = (char *)calloc((size_t)nthreads, sizeof(char));
    if (!joinable) {
        free(tids);
        free(args);
        convolve_rows_range(in, kern, out, 0, h);
        return;
    }

    /* Explicit contiguous row-range partitioning. */
    int base = h / nthreads;
    int rem = h % nthreads;
    int y = 0;
    int launched = 0;
    for (int t = 0; t < nthreads; ++t) {
        int rows = base + (t < rem ? 1 : 0);
        args[t].in = in;
        args[t].kern = kern;
        args[t].out = out;
        args[t].y0 = y;
        args[t].y1 = y + rows;
        y += rows;
        if (plat_thread_create(&tids[t], pthr_worker, &args[t]) == 0) {
            joinable[t] = 1;
            ++launched;
        } else {
            /* Could not spawn: run this range inline. */
            convolve_rows_range(in, kern, out, args[t].y0, args[t].y1);
        }
    }
    for (int t = 0; t < nthreads; ++t) {
        if (joinable[t])
            plat_thread_join(tids[t]);
    }
    (void)launched;
    free(joinable);
    free(tids);
    free(args);
}

/* ------------------------------------------------------------- OpenMP rows */

void convolve_openmp_rows(const image_t *in, const kernel_t *kern,
                          image_t *out, conv_sched_t sched, int chunk)
{
#ifdef _OPENMP
    omp_sched_t os;
    switch (sched) {
    case CONV_SCHED_DYNAMIC: os = omp_sched_dynamic; break;
    case CONV_SCHED_GUIDED:  os = omp_sched_guided;  break;
    case CONV_SCHED_STATIC:
    default:                 os = omp_sched_static;  break;
    }
    /* chunk <= 0 lets OpenMP pick its default chunk for the schedule. */
    omp_set_schedule(os, chunk > 0 ? chunk : 0);

    const int w = in->width;
    const int h = in->height;
    const int ch = in->channels;

    #pragma omp parallel for schedule(runtime)
    for (int oy = 0; oy < h; ++oy) {
        for (int ox = 0; ox < w; ++ox) {
            for (int c = 0; c < ch; ++c) {
                out->data[((size_t)oy * w + ox) * ch + c] =
                    convolve_pixel(in, kern, ox, oy, c);
            }
        }
    }
#else
    (void)sched;
    (void)chunk;
    convolve_rows_range(in, kern, out, 0, in->height);
#endif
}

/* ------------------------------------------------------------ OpenMP tiled */

void convolve_openmp_tiled(const image_t *in, const kernel_t *kern,
                           image_t *out, int tile)
{
    const int w = in->width;
    const int h = in->height;
    const int ch = in->channels;
    if (tile < 1)
        tile = 32;

    const int ntx = (w + tile - 1) / tile;
    const int nty = (h + tile - 1) / tile;

#ifdef _OPENMP
    #pragma omp parallel for collapse(2) schedule(static)
    for (int ty = 0; ty < nty; ++ty) {
        for (int tx = 0; tx < ntx; ++tx) {
            int y0 = ty * tile;
            int x0 = tx * tile;
            int y1 = y0 + tile; if (y1 > h) y1 = h;
            int x1 = x0 + tile; if (x1 > w) x1 = w;
            for (int oy = y0; oy < y1; ++oy) {
                for (int ox = x0; ox < x1; ++ox) {
                    for (int c = 0; c < ch; ++c) {
                        out->data[((size_t)oy * w + ox) * ch + c] =
                            convolve_pixel(in, kern, ox, oy, c);
                    }
                }
            }
        }
    }
#else
    for (int ty = 0; ty < nty; ++ty) {
        for (int tx = 0; tx < ntx; ++tx) {
            int y0 = ty * tile;
            int x0 = tx * tile;
            int y1 = y0 + tile; if (y1 > h) y1 = h;
            int x1 = x0 + tile; if (x1 > w) x1 = w;
            for (int oy = y0; oy < y1; ++oy) {
                for (int ox = x0; ox < x1; ++ox) {
                    for (int c = 0; c < ch; ++c) {
                        out->data[((size_t)oy * w + ox) * ch + c] =
                            convolve_pixel(in, kern, ox, oy, c);
                    }
                }
            }
        }
    }
#endif
}

/* --------------------------------------------------------------- validate */

validate_result_t validate(const image_t *a, const image_t *b, double epsilon)
{
    validate_result_t res;
    res.pass = 0;
    res.max_abs_diff = 0.0;
    res.first_mismatch = (size_t)-1;
    res.mm_x = res.mm_y = res.mm_c = -1;

    if (!a || !b || !a->data || !b->data ||
        a->width != b->width || a->height != b->height ||
        a->channels != b->channels) {
        return res; /* mismatched shapes => fail */
    }

    const size_t n = image_num_samples(a);
    const int w = a->width;
    const int ch = a->channels;
    double maxd = 0.0;
    size_t first = (size_t)-1;

    for (size_t i = 0; i < n; ++i) {
        double d = fabs((double)a->data[i] - (double)b->data[i]);
        if (d > maxd)
            maxd = d;
        if (d > epsilon && first == (size_t)-1)
            first = i;
    }

    res.max_abs_diff = maxd;
    res.first_mismatch = first;
    if (first != (size_t)-1) {
        size_t pix = first / (size_t)ch;
        res.mm_c = (int)(first % (size_t)ch);
        res.mm_x = (int)(pix % (size_t)w);
        res.mm_y = (int)(pix / (size_t)w);
    }
    res.pass = (maxd <= epsilon) ? 1 : 0;
    return res;
}


Writing /content/Artigo_PCD-main/src/convolution.c


In [6]:
%%writefile /content/Artigo_PCD-main/src/convolution.h
/*
 * convolution.h - Four 2D convolution implementations sharing one output
 * contract, plus pixel-by-pixel validation.
 *
 * Implementations:
 *   (a) convolve_sequential   - reference, compiles WITHOUT -fopenmp
 *   (b) convolve_pthreads     - explicit contiguous output-row ranges / thread
 *   (c) convolve_openmp_rows  - #pragma omp parallel for over outer row loop,
 *                               configurable schedule {static,dynamic,guided}
 *                               and chunk size (via omp_set_schedule +
 *                               schedule(runtime))
 *   (d) convolve_openmp_tiled - 2D block/tile decomposition (collapse(2) over
 *                               the tile grid)
 *
 * BORDER POLICY (shared by all four, identical => outputs match exactly):
 *   Zero-padding. Samples whose source coordinate falls outside the image are
 *   treated as 0. This is applied uniformly, so the sequential reference and
 *   every parallel variant produce bit-for-bit identical results, enabling
 *   pixel-by-pixel validation.
 *
 * The output image must be pre-allocated with the SAME dimensions/channels as
 * the input. Only the convolution math is performed here (no allocation, no
 * I/O) so the harness can time this path in isolation.
 */
#ifndef CONVOLUTION_H
#define CONVOLUTION_H

#include "image.h"
#include "kernels.h"

typedef enum {
    CONV_SCHED_STATIC = 0,
    CONV_SCHED_DYNAMIC = 1,
    CONV_SCHED_GUIDED = 2
} conv_sched_t;

/* Reference, single-threaded. Compilable without -fopenmp. */
void convolve_sequential(const image_t *in, const kernel_t *kern,
                         image_t *out);

/* Explicit Pthreads partitioning of output rows across `nthreads` threads. */
void convolve_pthreads(const image_t *in, const kernel_t *kern,
                       image_t *out, int nthreads);

/*
 * OpenMP parallel-for over the outer row loop. `sched`/`chunk` select the
 * runtime schedule (static/dynamic/guided). If chunk <= 0 the OpenMP default
 * chunk is used.
 */
void convolve_openmp_rows(const image_t *in, const kernel_t *kern,
                          image_t *out, conv_sched_t sched, int chunk);

/* OpenMP 2D tiled decomposition over a tile grid (collapse(2)). */
void convolve_openmp_tiled(const image_t *in, const kernel_t *kern,
                           image_t *out, int tile);

/* Result of a pixel-by-pixel comparison. */
typedef struct {
    int    pass;            /* 1 if max_abs_diff <= epsilon */
    double max_abs_diff;    /* largest |a-b| over all samples */
    size_t first_mismatch;  /* flat sample index of first > epsilon diff */
    int    mm_x, mm_y, mm_c; /* decoded coordinate of first mismatch (-1 none) */
} validate_result_t;

/*
 * Compare two images sample-by-sample within `epsilon`. Dimensions/channels
 * must match. Returns pass=1 when every |a-b| <= epsilon.
 */
validate_result_t validate(const image_t *a, const image_t *b, double epsilon);

#endif /* CONVOLUTION_H */


Writing /content/Artigo_PCD-main/src/convolution.h


In [7]:
%%writefile /content/Artigo_PCD-main/src/energy.c
/*
 * energy.c - RAPL energy sampling with a TDP-based estimate fallback.
 *
 * See energy.h for the overall contract. The RAPL path reads every
 * /sys/class/powercap/intel-rapl:*\/energy_uj domain it can open (package and
 * DRAM). Each top-level domain is tracked INDIVIDUALLY in an energy_sample_t
 * so that counter wraparound can be corrected per domain: when a single
 * domain's end counter is below its start counter, only that domain's own
 * max_energy_range_uj is added back, instead of the summed range of all
 * domains (which would over-correct when only one domain wrapped). If no
 * domain can be read, the sample is flagged rapl_ok = 0 and the caller's
 * energy_compute() uses the TDP estimate instead.
 */
#if !defined(_WIN32) && !defined(_GNU_SOURCE)
#define _GNU_SOURCE   /* syscall() for perf_event_open under -std=c11 */
#endif

#include "energy.h"
#include "platform.h"

#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#if !defined(_WIN32)
/*
 * RAPL energy sampling via the Linux powercap sysfs interface. This whole
 * implementation is Linux-only; on Windows (see the stub below) the probe
 * returns 0 so energy_snapshot sets rapl_ok=0 and energy_compute uses the
 * UNCHANGED TDP_ESTIMATE fallback.
 */
#include <dirent.h>

#define RAPL_ROOT "/sys/class/powercap"

/* Read one unsigned-long value from a sysfs file. Returns 0 on success. */
static int read_u64_file(const char *path, uint64_t *out)
{
    FILE *f = fopen(path, "r");
    if (!f) return -1;
    unsigned long long v = 0;
    int n = fscanf(f, "%llu", &v);
    fclose(f);
    if (n != 1) return -1;
    *out = (uint64_t)v;
    return 0;
}


/* ----------------------------------------------------------------------
 * Backend 2 (Linux): RAPL through perf_event_open on the "power" PMU.
 *
 * The powercap sysfs tree (/sys/class/powercap/intel-rapl*) is only populated
 * on some kernels/CPUs. The same hardware counters are exposed to userspace
 * by the "power" perf PMU (event power/energy-pkg/), which is also what
 * `perf stat -e power/energy-pkg/` uses. This covers AMD Zen (e.g. Ryzen)
 * as well as Intel. It needs either root/CAP_PERFMON or
 * /proc/sys/kernel/perf_event_paranoid <= 0, and it does NOT work inside most
 * containers/VMs (the usual case on Google Colab) -- in that case the open
 * fails and the caller falls back to TDP_ESTIMATE, exactly as before.
 * ---------------------------------------------------------------------- */
#include <linux/perf_event.h>
#include <sys/syscall.h>
#include <unistd.h>

#define PERF_PMU "/sys/bus/event_source/devices/power"

static int    g_perf_state = 0;     /* 0 = untried, 1 = ok, -1 = unavailable */
static int    g_perf_fd = -1;
static double g_perf_scale_j = 0.0; /* Joules per counter tick               */

static int perf_try_open(void)
{
    if (g_perf_state != 0) return g_perf_state == 1;
    g_perf_state = -1;

    FILE *f;
    unsigned long long type = 0, config = 0;
    double scale = 0.0;
    int cpu = 0;

    f = fopen(PERF_PMU "/type", "r");
    if (!f) return 0;
    if (fscanf(f, "%llu", &type) != 1) { fclose(f); return 0; }
    fclose(f);

    f = fopen(PERF_PMU "/events/energy-pkg", "r");   /* e.g. "event=0x02" */
    if (!f) return 0;
    if (fscanf(f, "event=%llx", &config) != 1) { fclose(f); return 0; }
    fclose(f);

    f = fopen(PERF_PMU "/events/energy-pkg.scale", "r");
    if (!f) return 0;
    if (fscanf(f, "%lf", &scale) != 1 || scale <= 0.0) { fclose(f); return 0; }
    fclose(f);

    /* The PMU is per-package; open it on the first CPU of its cpumask. */
    f = fopen(PERF_PMU "/cpumask", "r");
    if (f) {
        int c;
        if (fscanf(f, "%d", &c) == 1 && c >= 0) cpu = c;
        fclose(f);
    }

    struct perf_event_attr attr;
    memset(&attr, 0, sizeof(attr));
    attr.type = (uint32_t)type;
    attr.size = sizeof(attr);
    attr.config = (uint64_t)config;

    long fd = syscall(__NR_perf_event_open, &attr, -1, cpu, -1, 0);
    if (fd < 0) return 0;

    g_perf_fd = (int)fd;
    g_perf_scale_j = scale;
    g_perf_state = 1;
    return 1;
}

/* One synthetic "domain": the package energy counter in microjoules. The
 * kernel already accumulates it in 64 bits, so range_uj = 0 (no wraparound
 * correction needed). Returns 1 if read, 0 otherwise. */
static int perf_read_domain(rapl_domain_sample_t *out)
{
    if (!perf_try_open()) return 0;
    uint64_t ticks = 0;
    if (read(g_perf_fd, &ticks, sizeof(ticks)) != (ssize_t)sizeof(ticks))
        return 0;
    out->energy_uj = (uint64_t)((double)ticks * g_perf_scale_j * 1e6);
    out->range_uj = 0;
    return 1;
}

static const char *g_backend = "none";

/*
 * Read the energy_uj and max_energy_range_uj of each top-level intel-rapl
 * domain INDIVIDUALLY into `out` (up to ENERGY_MAX_RAPL_DOMAINS). Only the
 * top-level domains (intel-rapl:N) are read to avoid double-counting the
 * nested sub-domains (intel-rapl:N:M) which are already part of the parent.
 * Keeping domains separate lets energy_compute() correct counter wraparound
 * per domain instead of adding the summed range of every domain. The read
 * order is deterministic enough within a single process that domain i in the
 * start snapshot corresponds to domain i in the end snapshot; to be robust we
 * do not rely on that and simply pair by index since the directory contents do
 * not change between the two snapshots.
 * Returns the number of domains read (0 means RAPL is unavailable here).
 */
static int rapl_read_domains(rapl_domain_sample_t *out, int max_domains)
{
    DIR *d = opendir(RAPL_ROOT);
    if (!d) {
        if (max_domains > 0 && perf_read_domain(&out[0])) {
            g_backend = "perf_event (power/energy-pkg)";
            return 1;
        }
        return 0;
    }

    int found = 0;

    struct dirent *de;
    while ((de = readdir(d)) != NULL && found < max_domains) {
        /* Top-level packages are named "intel-rapl:0", "intel-rapl:1", ...
         * Sub-domains contain a second colon ("intel-rapl:0:0") and are
         * skipped to avoid double counting. */
        if (strncmp(de->d_name, "intel-rapl:", 11) != 0) continue;
        if (strchr(de->d_name + 11, ':') != NULL) continue;

        char path[512];
        uint64_t e = 0, r = 0;

        snprintf(path, sizeof(path), RAPL_ROOT "/%s/energy_uj", de->d_name);
        if (read_u64_file(path, &e) != 0) continue;

        snprintf(path, sizeof(path), RAPL_ROOT "/%s/max_energy_range_uj",
                 de->d_name);
        if (read_u64_file(path, &r) != 0) r = 0;

        out[found].energy_uj = e;
        out[found].range_uj = r;
        found++;
    }
    closedir(d);

    if (found > 0) {
        g_backend = "powercap (intel-rapl sysfs)";
    } else if (max_domains > 0 && perf_read_domain(&out[0])) {
        found = 1;
        g_backend = "perf_event (power/energy-pkg)";
    }
    return found;
}

#else /* _WIN32 */

/*
 * Windows has no RAPL powercap sysfs interface, so the probe reports "no
 * domains available". This forces rapl_ok=0 and the existing TDP_ESTIMATE
 * fallback in energy_compute, keeping all energy labelling/caveats intact.
 */
static const char *g_backend = "none";

static int rapl_read_domains(rapl_domain_sample_t *out, int max_domains)
{
    (void)out;
    (void)max_domains;
    return 0;
}

#endif /* !_WIN32 */

/* Process CPU time (user + system) in nanoseconds. */
static uint64_t process_cpu_time_ns(void)
{
    return plat_process_cpu_time_ns();
}

const char *energy_source_label(energy_source_t src)
{
    switch (src) {
    case ENERGY_SOURCE_RAPL:         return "RAPL";
    case ENERGY_SOURCE_TDP_ESTIMATE: return "TDP_ESTIMATE";
    default:                         return "UNKNOWN";
    }
}

static void energy_snapshot(energy_sample_t *out)
{
    memset(out, 0, sizeof(*out));
    out->ndomains = rapl_read_domains(out->domains, ENERGY_MAX_RAPL_DOMAINS);
    out->rapl_ok = (out->ndomains > 0) ? 1 : 0;
    out->cpu_time_ns = process_cpu_time_ns();
}

const char *energy_probe(void)
{
    energy_sample_t tmp;
    energy_snapshot(&tmp);
    return tmp.rapl_ok ? g_backend : "none (falls back to TDP_ESTIMATE)";
}

void energy_begin(energy_sample_t *out) { energy_snapshot(out); }
void energy_end(energy_sample_t *out)   { energy_snapshot(out); }

energy_result_t energy_compute(const energy_sample_t *start,
                               const energy_sample_t *end,
                               double wall_seconds,
                               double tdp_watts)
{
    energy_result_t r;
    memset(&r, 0, sizeof(r));

    if (wall_seconds <= 0.0) wall_seconds = 1e-9;

    /* CPU utilization in "busy cores" units: delta CPU time / wall time. */
    double cpu_delta_ns = (double)end->cpu_time_ns - (double)start->cpu_time_ns;
    if (cpu_delta_ns < 0.0) cpu_delta_ns = 0.0;
    double busy_cores = (cpu_delta_ns / 1e9) / wall_seconds;
    r.cpu_utilization = busy_cores;

    if (start->rapl_ok && end->rapl_ok) {
        /* Measured path: sum the PER-DOMAIN deltas, correcting wraparound for
         * each domain independently. A single domain whose end counter is
         * below its start counter wrapped once; we add back only THAT domain's
         * own max_energy_range_uj, never the summed range of all domains, so a
         * single wrap cannot over-correct the others. Domains are paired by
         * index because the powercap directory does not change between the two
         * snapshots. */
        int ndom = (start->ndomains < end->ndomains)
                 ? start->ndomains : end->ndomains;
        double total_uj = 0.0;
        for (int i = 0; i < ndom; ++i) {
            double delta_uj = (double)end->domains[i].energy_uj
                            - (double)start->domains[i].energy_uj;
            if (delta_uj < 0.0) {
                double range = (double)start->domains[i].range_uj;
                if (range > 0.0) delta_uj += range;
            }
            total_uj += delta_uj;
        }
        r.source = ENERGY_SOURCE_RAPL;
        r.energy_j = total_uj / 1e6;
    } else {
        /* Estimated path: average busy cores * nominal per-core-equivalent TDP
         * share * wall time. tdp_watts is treated as the full-package power at
         * full utilization of all cores; we scale it by the measured busy-core
         * fraction relative to the online CPU count so a single busy core does
         * not get charged the whole package TDP.
         *
         * CAVEAT: in this branch avg_power_w reduces to
         *   tdp_watts * (busy_cores / ncpu)
         * (wall time cancels), so power here is a pure function of average
         * busy-core occupancy, NOT a measured energy difference between models.
         * Any MFLOP/W derived from it is therefore only an occupancy-normalized
         * efficiency proxy WITHIN a single model; it is NOT comparable ACROSS
         * programming models (a model that keeps more cores accounted as busy
         * reports proportionally more watts). The harness labels every such row
         * TDP_ESTIMATE and the README states this caveat prominently. */
        long ncpu = plat_online_cpus();
        if (ncpu < 1) ncpu = 1;
        double util_fraction = busy_cores / (double)ncpu;
        if (util_fraction > 1.0) util_fraction = 1.0;
        r.source = ENERGY_SOURCE_TDP_ESTIMATE;
        r.energy_j = tdp_watts * util_fraction * wall_seconds;
    }

    r.avg_power_w = r.energy_j / wall_seconds;
    return r;
}


Writing /content/Artigo_PCD-main/src/energy.c


In [8]:
%%writefile /content/Artigo_PCD-main/src/energy.h
/*
 * energy.h - Energy accounting for a timed region.
 *
 * Two sources are supported, probed in this order:
 *
 *   (1) ENERGY_SOURCE_RAPL      - Intel RAPL via the powercap sysfs interface
 *                                 (/sys/class/powercap/intel-rapl/.../energy_uj).
 *                                 Package + DRAM domains are read individually;
 *                                 the counter wraparound (max_energy_range_uj)
 *                                 is handled PER DOMAIN so a single domain
 *                                 wrapping does not over-correct the others.
 *
 *   (2) ENERGY_SOURCE_TDP_ESTIMATE - When RAPL is absent or unreadable (the
 *                                 case on this sandbox), fall back to a
 *                                 TDP-based ESTIMATE. We measure process CPU
 *                                 time (utime+stime) over the wall-clock span
 *                                 to obtain the average number of busy cores,
 *                                 then estimate Joules as:
 *                                     E ~= tdp_watts * busy_cores_fraction
 *                                          * wall_seconds
 *                                 This is an ESTIMATE, never a measurement, and
 *                                 is labelled TDP_ESTIMATE everywhere it is
 *                                 printed or written.
 *
 * Usage:
 *   energy_sample_t s0, s1;
 *   energy_begin(&s0);
 *   ... work ...
 *   energy_end(&s1);
 *   energy_result_t r = energy_compute(&s0, &s1, wall_seconds, tdp_watts);
 */
#ifndef ENERGY_H
#define ENERGY_H

#include <stdint.h>

typedef enum {
    ENERGY_SOURCE_RAPL = 0,         /* directly measured via RAPL counters   */
    ENERGY_SOURCE_TDP_ESTIMATE = 1  /* estimated from CPU utilization * TDP   */
} energy_source_t;

/* Maximum number of top-level RAPL domains tracked per snapshot. */
#define ENERGY_MAX_RAPL_DOMAINS 16

/* Per-domain RAPL reading captured in a snapshot. */
typedef struct {
    uint64_t energy_uj;      /* this domain's energy_uj counter at snapshot  */
    uint64_t range_uj;       /* this domain's max_energy_range_uj (0 if n/a) */
} rapl_domain_sample_t;

/* A snapshot taken at the start/end of a timed region. */
typedef struct {
    int      rapl_ok;        /* 1 if RAPL energy counters were read          */
    int      ndomains;       /* number of top-level domains captured         */
    rapl_domain_sample_t domains[ENERGY_MAX_RAPL_DOMAINS];
    uint64_t cpu_time_ns;    /* process CPU time (utime+stime) in nanoseconds */
} energy_sample_t;

/* Outcome of comparing two snapshots over a known wall-clock interval. */
typedef struct {
    energy_source_t source;  /* RAPL vs TDP estimate                         */
    double          energy_j;      /* energy over the interval, Joules       */
    double          avg_power_w;   /* energy_j / wall_seconds                */
    double          cpu_utilization; /* busy-core fraction (0..ncpus)        */
} energy_result_t;

/* Short label for a source, e.g. "RAPL" or "TDP_ESTIMATE". */
const char *energy_source_label(energy_source_t src);

/* Human-readable name of the energy backend that will actually be used
 * ("powercap ...", "perf_event ...", or "none ..."). Probes once. */
const char *energy_probe(void);

/* Take a snapshot (RAPL energy if available, plus process CPU time). */
void energy_begin(energy_sample_t *out);
void energy_end(energy_sample_t *out);

/*
 * Compute energy for the interval [start,end] that spanned `wall_seconds`.
 * If both snapshots have valid RAPL readings, uses the measured delta;
 * otherwise falls back to the TDP estimate using `tdp_watts`.
 */
energy_result_t energy_compute(const energy_sample_t *start,
                               const energy_sample_t *end,
                               double wall_seconds,
                               double tdp_watts);

#endif /* ENERGY_H */


Writing /content/Artigo_PCD-main/src/energy.h


In [9]:
%%writefile /content/Artigo_PCD-main/src/image.c
/*
 * image.c - Image container, synthetic generation, binary PGM/PPM I/O.
 */
#include "image.h"

#include <stdio.h>
#include <stdlib.h>
#include <string.h>

int image_alloc(image_t *img, int width, int height, int channels)
{
    if (!img || width <= 0 || height <= 0 || (channels != 1 && channels != 3))
        return 1;
    size_t n = (size_t)width * (size_t)height * (size_t)channels;
    float *data = (float *)calloc(n, sizeof(float));
    if (!data)
        return 2;
    img->width = width;
    img->height = height;
    img->channels = channels;
    img->data = data;
    return 0;
}

void image_free(image_t *img)
{
    if (!img)
        return;
    free(img->data);
    img->data = NULL;
    img->width = img->height = img->channels = 0;
}

/* Small, fast, deterministic PRNG (xorshift32). */
static inline uint32_t xorshift32(uint32_t *state)
{
    uint32_t x = *state;
    x ^= x << 13;
    x ^= x >> 17;
    x ^= x << 5;
    *state = x;
    return x;
}

int image_generate(image_t *img, int width, int height, int channels,
                   uint32_t seed)
{
    int rc = image_alloc(img, width, height, channels);
    if (rc)
        return rc;

    uint32_t state = seed ? seed : 0x9E3779B9u; /* avoid zero state */
    const int block = 32; /* checkerboard block size for strong edges */

    for (int y = 0; y < height; ++y) {
        for (int x = 0; x < width; ++x) {
            /* Structured component: diagonal gradient + checkerboard. */
            float grad = 255.0f * ((float)(x + y) /
                                   (float)(width + height - 2 > 0
                                               ? (width + height - 2)
                                               : 1));
            int checker = (((x / block) + (y / block)) & 1) ? 1 : 0;
            float base = 0.6f * grad + (checker ? 80.0f : 0.0f);

            for (int c = 0; c < channels; ++c) {
                /* Reproducible noise in [-20, 20). */
                float noise =
                    ((float)(xorshift32(&state) & 0xFFFF) / 65535.0f) * 40.0f
                    - 20.0f;
                /* Per-channel tint so color images differ across channels. */
                float tint = (channels == 3) ? (float)(c * 20) : 0.0f;
                float v = base + noise + tint;
                if (v < 0.0f) v = 0.0f;
                if (v > 255.0f) v = 255.0f;
                img->data[((size_t)y * width + x) * channels + c] = v;
            }
        }
    }
    return 0;
}

/* Clamp+round a float sample to an 8-bit value. */
static inline unsigned char to_u8(float v)
{
    if (v < 0.0f) v = 0.0f;
    if (v > 255.0f) v = 255.0f;
    return (unsigned char)(v + 0.5f);
}

/* Skip whitespace and '#' comment lines in a PNM header. */
static int pnm_skip_ws(FILE *f)
{
    int c;
    for (;;) {
        c = fgetc(f);
        if (c == '#') {
            while ((c = fgetc(f)) != '\n' && c != EOF)
                ;
        } else if (c == ' ' || c == '\t' || c == '\n' || c == '\r') {
            continue;
        } else {
            if (c != EOF)
                ungetc(c, f);
            return 0;
        }
    }
}

static int write_pnm(const image_t *img, const char *path, int channels,
                     const char *magic)
{
    if (!img || img->channels != channels || !img->data)
        return 1;
    FILE *f = fopen(path, "wb");
    if (!f)
        return 2;

    fprintf(f, "%s\n%d %d\n255\n", magic, img->width, img->height);

    size_t n = image_num_samples(img);
    unsigned char *buf = (unsigned char *)malloc(n);
    if (!buf) {
        fclose(f);
        return 3;
    }
    for (size_t i = 0; i < n; ++i)
        buf[i] = to_u8(img->data[i]);

    size_t wrote = fwrite(buf, 1, n, f);
    free(buf);
    fclose(f);
    return (wrote == n) ? 0 : 4;
}

static int read_pnm(image_t *img, const char *path, int channels,
                    const char *magic)
{
    FILE *f = fopen(path, "rb");
    if (!f)
        return 1;

    char m[3] = {0};
    if (fscanf(f, "%2s", m) != 1 || strcmp(m, magic) != 0) {
        fclose(f);
        return 2;
    }
    int w = 0, h = 0, maxv = 0;
    pnm_skip_ws(f);
    if (fscanf(f, "%d", &w) != 1) { fclose(f); return 3; }
    pnm_skip_ws(f);
    if (fscanf(f, "%d", &h) != 1) { fclose(f); return 3; }
    pnm_skip_ws(f);
    if (fscanf(f, "%d", &maxv) != 1 || maxv != 255) { fclose(f); return 3; }
    /* Exactly one whitespace byte separates header from binary data. */
    fgetc(f);

    if (image_alloc(img, w, h, channels)) {
        fclose(f);
        return 4;
    }
    size_t n = image_num_samples(img);
    unsigned char *buf = (unsigned char *)malloc(n);
    if (!buf) {
        image_free(img);
        fclose(f);
        return 5;
    }
    size_t got = fread(buf, 1, n, f);
    fclose(f);
    if (got != n) {
        free(buf);
        image_free(img);
        return 6;
    }
    for (size_t i = 0; i < n; ++i)
        img->data[i] = (float)buf[i];
    free(buf);
    return 0;
}

int image_write_pgm(const image_t *img, const char *path)
{
    return write_pnm(img, path, 1, "P5");
}

int image_read_pgm(image_t *img, const char *path)
{
    return read_pnm(img, path, 1, "P5");
}

int image_write_ppm(const image_t *img, const char *path)
{
    return write_pnm(img, path, 3, "P6");
}

int image_read_ppm(image_t *img, const char *path)
{
    return read_pnm(img, path, 3, "P6");
}


Writing /content/Artigo_PCD-main/src/image.c


In [10]:
%%writefile /content/Artigo_PCD-main/src/image.h
/*
 * image.h - Float-backed image container, synthetic generation and binary
 * PGM (P5, grayscale) / PPM (P6, color) I/O.
 *
 * The image data is stored as float in [0, 255] (pixel values kept in the
 * same scale as 8-bit samples so convolution math is straightforward and
 * round-trips through 8-bit PGM/PPM cleanly). Channel-interleaved, row-major:
 *   data[(y*width + x)*channels + c]
 *
 * Image generation and file I/O are deliberately kept SEPARATE from the timed
 * convolution kernels so the benchmark harness can exclude them from timing.
 */
#ifndef IMAGE_H
#define IMAGE_H

#include <stddef.h>
#include <stdint.h>

typedef struct {
    int    width;
    int    height;
    int    channels; /* 1 = grayscale (PGM), 3 = color (PPM) */
    float *data;     /* width*height*channels floats, values in [0,255] */
} image_t;

/* Allocate an image (data zeroed). Returns 0 on success. */
int image_alloc(image_t *img, int width, int height, int channels);

/* Free image data (safe on zeroed struct). */
void image_free(image_t *img);

/* Number of float samples in the image. */
static inline size_t image_num_samples(const image_t *img)
{
    return (size_t)img->width * (size_t)img->height * (size_t)img->channels;
}

/*
 * Generate a synthetic, deterministic image seeded by `seed`. Combines a
 * structured component (diagonal gradient + checkerboard blocks) so that
 * strong edges exist for Sobel, with reproducible pseudo-random noise so the
 * content is non-trivial. Same seed + dimensions => identical bytes.
 */
int image_generate(image_t *img, int width, int height, int channels,
                   uint32_t seed);

/*
 * Binary PGM (P5) write/read. Requires channels == 1. Values are clamped and
 * rounded to 8-bit on write; read produces floats in [0,255].
 */
int image_write_pgm(const image_t *img, const char *path);
int image_read_pgm(image_t *img, const char *path);

/* Binary PPM (P6) write/read. Requires channels == 3. */
int image_write_ppm(const image_t *img, const char *path);
int image_read_ppm(image_t *img, const char *path);

#endif /* IMAGE_H */


Writing /content/Artigo_PCD-main/src/image.h


In [11]:
%%writefile /content/Artigo_PCD-main/src/kernels.c
/*
 * kernels.c - Convolution kernel construction.
 *
 * See kernels.h for the pattern/size contract and the note on the SOBEL
 * linear-convolution choice.
 */
#include "kernels.h"

#include <math.h>
#include <stdlib.h>
#include <string.h>
#include <stdio.h>

const char *kernel_pattern_name(conv_pattern_t p)
{
    switch (p) {
    case PATTERN_SOBEL:    return "sobel";
    case PATTERN_GAUSSIAN: return "gaussian";
    case PATTERN_BOX:      return "box";
    default:               return "unknown";
    }
}

static int size_supported(conv_pattern_t pattern, int k)
{
    if (k < 1 || (k % 2) == 0)
        return 0; /* must be odd */
    if (pattern == PATTERN_SOBEL)
        return (k == 3);
    /* GAUSSIAN and BOX */
    return (k == 3 || k == 7 || k == 11 || k == 15);
}

/* Classic Sobel horizontal gradient Gx (3x3). */
static void fill_sobel3(float *d)
{
    const float gx[9] = {
        -1.f, 0.f, 1.f,
        -2.f, 0.f, 2.f,
        -1.f, 0.f, 1.f
    };
    memcpy(d, gx, sizeof(gx));
}

/*
 * Gaussian kernel of dimension k, normalized to sum 1.
 * sigma is chosen from the radius so the window captures ~3 sigma:
 *   sigma = (k/2) / 3  = radius / 3.  Documented, deterministic choice.
 */
static void fill_gaussian(float *d, int k)
{
    const int r = k / 2;
    const double sigma = (r > 0) ? ((double)r / 3.0) : 1.0;
    const double s2 = 2.0 * sigma * sigma;
    double sum = 0.0;
    for (int y = -r; y <= r; ++y) {
        for (int x = -r; x <= r; ++x) {
            double v = exp(-((double)x * x + (double)y * y) / s2);
            d[(y + r) * k + (x + r)] = (float)v;
            sum += v;
        }
    }
    /* Normalize so coefficients sum to 1 (smoothing preserves brightness). */
    for (int i = 0; i < k * k; ++i)
        d[i] = (float)((double)d[i] / sum);
}

/* Box / mean kernel: every coefficient = 1/(k*k). */
static void fill_box(float *d, int k)
{
    const float v = 1.0f / (float)(k * k);
    for (int i = 0; i < k * k; ++i)
        d[i] = v;
}

int kernel_build(conv_pattern_t pattern, int k, kernel_t *out)
{
    if (!out || !size_supported(pattern, k))
        return 1;

    float *data = (float *)malloc((size_t)k * (size_t)k * sizeof(float));
    if (!data)
        return 2;

    switch (pattern) {
    case PATTERN_SOBEL:    fill_sobel3(data);  break;
    case PATTERN_GAUSSIAN: fill_gaussian(data, k); break;
    case PATTERN_BOX:      fill_box(data, k); break;
    default:
        free(data);
        return 3;
    }

    out->data = data;
    out->k = k;
    out->pattern = pattern;
    snprintf(out->name, sizeof(out->name), "%s-%dx%d",
             kernel_pattern_name(pattern), k, k);
    return 0;
}

void kernel_free(kernel_t *kern)
{
    if (!kern)
        return;
    free(kern->data);
    kern->data = NULL;
    kern->k = 0;
}


Writing /content/Artigo_PCD-main/src/kernels.c


In [12]:
%%writefile /content/Artigo_PCD-main/src/kernels.h
/*
 * kernels.h - Convolution kernel (filter) definitions.
 *
 * Exposes the 3 convolution PATTERNS that form the explicit experiment axis
 * of this benchmark:
 *   (1) SOBEL    - edge detection (3x3 only; linear horizontal Gx, NOT the
 *                  combined gradient magnitude -- see the note below)
 *   (2) GAUSSIAN - smoothing (3x3, 7x7, 11x11, 15x15)
 *   (3) BOX      - mean / averaging blur (3x3, 7x7, 11x11, 15x15)
 *
 * A kernel is returned as a flat float buffer of size k*k (row-major) together
 * with its size k and a human-readable name. Caller owns the buffer and must
 * free it with kernel_free().
 */
#ifndef KERNELS_H
#define KERNELS_H

#include <stddef.h>

typedef enum {
    PATTERN_SOBEL = 0,   /* edge detection */
    PATTERN_GAUSSIAN = 1, /* smoothing */
    PATTERN_BOX = 2       /* mean / averaging */
} conv_pattern_t;

#define CONV_NUM_PATTERNS 3

/*
 * A convolution kernel: flat row-major coefficients, dimension k (k x k),
 * and a descriptive name (e.g. "gaussian-7x7").
 *
 * Note on SOBEL: because the Sobel operator is a gradient magnitude built from
 * two directional kernels (Gx, Gy), it is NOT a single linear kernel. For the
 * generic coefficient-returning API we expose the horizontal Gx 3x3 kernel so
 * all code paths can treat every pattern uniformly as a single linear
 * convolution. The documented, consistent behaviour across ALL four
 * implementations is therefore a plain linear convolution with these
 * coefficients; this keeps the four implementations bit-for-bit comparable,
 * which is the whole point of validation. (A true magnitude filter would need
 * two passes; that is out of scope for the shared linear-convolution contract.)
 */
typedef struct {
    float *data;   /* k*k floats, row-major */
    int    k;      /* kernel dimension (odd) */
    char   name[64];
    conv_pattern_t pattern;
} kernel_t;

/* Human-readable pattern name, e.g. "sobel". */
const char *kernel_pattern_name(conv_pattern_t p);

/*
 * Build a kernel for (pattern, k). Returns 0 on success and fills *out.
 * Returns non-zero on an unsupported (pattern, size) combination.
 * Supported sizes:
 *   SOBEL:    k == 3 only
 *   GAUSSIAN: k in {3,7,11,15}
 *   BOX:      k in {3,7,11,15}
 * On success, out->data is heap-allocated; free with kernel_free().
 */
int kernel_build(conv_pattern_t pattern, int k, kernel_t *out);

/* Free the coefficient buffer owned by a kernel_t (safe on zeroed struct). */
void kernel_free(kernel_t *kern);

#endif /* KERNELS_H */


Writing /content/Artigo_PCD-main/src/kernels.h


In [13]:
%%writefile /content/Artigo_PCD-main/src/main.c
/*
 * main.c - Benchmark harness for the 2D convolution study.
 *
 * Drives the full experimental matrix over the article's axes plus the user's
 * three explicit extensions:
 *   - convolution PATTERN (sobel, gaussian, box)   [user axis #1]
 *   - IMAGE SIZE (2k, 4k, 8k; tiny in --quick)       [user axis #2]
 *   - MFLOP/W output metric                          [user axis #3]
 * together with kernel size, parallel model, thread count, OpenMP schedule and
 * chunk size.
 *
 * For every configuration it:
 *   1. generates the synthetic image ONCE per (pattern,size,image),
 *   2. computes the sequential reference ONCE per (pattern,size,image),
 *   3. runs `warmups` discarded warmup runs then `reps` TIMED runs
 *      (CLOCK_MONOTONIC around the convolve call ONLY),
 *   4. VALIDATES each parallel output pixel-by-pixel vs the reference and
 *      flags/skips any configuration that fails,
 *   5. aggregates statistics and records a CSV row + a stdout table line.
 *
 * Energy is sampled via RAPL when present; on this sandbox RAPL is absent so
 * the TDP-based ESTIMATE runs and is labelled TDP_ESTIMATE everywhere.
 *
 * Timing isolation: ONLY the convolve_* call is bracketed by the timer and the
 * energy sampler. Image generation, allocation and any I/O are outside it.
 */
#include "image.h"
#include "kernels.h"
#include "convolution.h"
#include "energy.h"
#include "metrics.h"
#include "platform.h"

#include <errno.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

#ifdef _OPENMP
#include <omp.h>
#endif

/* ---- Validation epsilon: outputs are bit-exact (see FEAT-001), tiny eps. -- */
#define VALIDATE_EPS 1e-3

/* ---- Model identifiers ---------------------------------------------------- */
typedef enum {
    MODEL_SEQ = 0,
    MODEL_PTHREADS = 1,
    MODEL_OMP_ROWS = 2,
    MODEL_OMP_TILED = 3,
    MODEL_COUNT = 4
} model_t;

static const char *model_name(model_t m)
{
    switch (m) {
    case MODEL_SEQ:       return "seq";
    case MODEL_PTHREADS:  return "pthreads";
    case MODEL_OMP_ROWS:  return "omp-rows";
    case MODEL_OMP_TILED: return "omp-tiled";
    default:              return "?";
    }
}

static const char *sched_name(conv_sched_t s)
{
    switch (s) {
    case CONV_SCHED_STATIC:  return "static";
    case CONV_SCHED_DYNAMIC: return "dynamic";
    case CONV_SCHED_GUIDED:  return "guided";
    default:                 return "?";
    }
}

/* ---- Image size axis ------------------------------------------------------ */
typedef struct { const char *name; int w, h, ch; } imgsize_t;

/* ---- Configuration gathered from the CLI --------------------------------- */
#define MAX_AXIS 16

typedef struct {
    conv_pattern_t patterns[MAX_AXIS]; int npatterns;
    int            sizes[MAX_AXIS];     int nsizes;
    imgsize_t      images[MAX_AXIS];    int nimages;
    int            threads[MAX_AXIS];   int nthreads;
    model_t        models[MAX_AXIS];    int nmodels;
    conv_sched_t   scheds[MAX_AXIS];    int nscheds;
    int            chunks[MAX_AXIS];    int nchunks;
    int            tile;        /* tile size for omp-tiled */
    int            reps;
    int            warmups;
    double         tdp_watts;
    const char    *out_path;
    uint32_t       seed;
    int            allow_oversubscribe;
} config_t;

/* ---- CSV header ----------------------------------------------------------- */
static const char *CSV_HEADER =
    "pattern,kernel_size,image,width,height,channels,model,threads,schedule,"
    "chunk,reps,time_mean_s,time_median_s,time_stddev_s,time_cv,speedup,"
    "efficiency,overhead_pct,flops,bytes_moved,arithmetic_intensity,energy_J,"
    "avg_power_W,EDP,MFLOP_per_W,cpu_utilization,energy_source,validation\n";

/* ------------------------------------------------------------------------- */
/* CLI parsing helpers                                                        */
/* ------------------------------------------------------------------------- */

static int parse_pattern(const char *s, conv_pattern_t *out)
{
    if (strcmp(s, "sobel") == 0)    { *out = PATTERN_SOBEL;    return 0; }
    if (strcmp(s, "gaussian") == 0) { *out = PATTERN_GAUSSIAN; return 0; }
    if (strcmp(s, "box") == 0)      { *out = PATTERN_BOX;      return 0; }
    return -1;
}

static int parse_model(const char *s, model_t *out)
{
    if (strcmp(s, "seq") == 0)       { *out = MODEL_SEQ;       return 0; }
    if (strcmp(s, "pthreads") == 0)  { *out = MODEL_PTHREADS;  return 0; }
    if (strcmp(s, "omp-rows") == 0)  { *out = MODEL_OMP_ROWS;  return 0; }
    if (strcmp(s, "omp-tiled") == 0) { *out = MODEL_OMP_TILED; return 0; }
    return -1;
}

static int parse_sched(const char *s, conv_sched_t *out)
{
    if (strcmp(s, "static") == 0)  { *out = CONV_SCHED_STATIC;  return 0; }
    if (strcmp(s, "dynamic") == 0) { *out = CONV_SCHED_DYNAMIC; return 0; }
    if (strcmp(s, "guided") == 0)  { *out = CONV_SCHED_GUIDED;  return 0; }
    return -1;
}

static int parse_image(const char *s, imgsize_t *out)
{
    if (strcmp(s, "2k") == 0) { out->name = "2k"; out->w = 2560; out->h = 1440; out->ch = 3; return 0; }
    if (strcmp(s, "4k") == 0) { out->name = "4k"; out->w = 3840; out->h = 2160; out->ch = 3; return 0; }
    if (strcmp(s, "8k") == 0) { out->name = "8k"; out->w = 7680; out->h = 4320; out->ch = 3; return 0; }
    return -1;
}

/* Portable strdup: strdup is POSIX, not ISO C, so we provide a tiny local
 * duplicate using only standard C (malloc/memcpy). This keeps main.c free of
 * any POSIX feature-test macro and compiles cleanly on Windows too. */
static char *dup_cstr(const char *s)
{
    size_t n = strlen(s) + 1;
    char *p = (char *)malloc(n);
    if (p) memcpy(p, s, n);
    return p;
}

/* Split a comma list and call `fn` for each token. Returns 0 on success. */
static int for_each_token(const char *list,
                          int (*fn)(const char *tok, void *ud), void *ud)
{
    char *copy = dup_cstr(list);
    if (!copy) return -1;
    int rc = 0;
    for (char *tok = strtok(copy, ","); tok; tok = strtok(NULL, ",")) {
        if (fn(tok, ud) != 0) { rc = -1; break; }
    }
    free(copy);
    return rc;
}

/*
 * Report (once) that an axis list is longer than MAX_AXIS and the extra token
 * is being dropped, so a long --threads/--chunks/... list never truncates
 * silently. Returns -1 so the caller aborts rather than running a quietly
 * truncated matrix.
 */
static int axis_overflow(const char *axis, const char *tok) {
    fprintf(stderr,
            "error: too many '%s' values (max %d); token '%s' and any after it "
            "exceed the limit. Reduce the list or raise MAX_AXIS.\n",
            axis, MAX_AXIS, tok);
    return -1;
}

/* collectors for for_each_token */
static int col_pattern(const char *t, void *ud) {
    config_t *c = ud; conv_pattern_t p;
    if (parse_pattern(t, &p) != 0) { fprintf(stderr, "unknown pattern '%s'\n", t); return -1; }
    if (c->npatterns >= MAX_AXIS) return axis_overflow("patterns", t);
    c->patterns[c->npatterns++] = p;
    return 0;
}
static int col_size(const char *t, void *ud) {
    config_t *c = ud; int k = atoi(t);
    if (k <= 0) { fprintf(stderr, "bad kernel size '%s'\n", t); return -1; }
    if (c->nsizes >= MAX_AXIS) return axis_overflow("sizes", t);
    c->sizes[c->nsizes++] = k;
    return 0;
}
static int col_image(const char *t, void *ud) {
    config_t *c = ud; imgsize_t im;
    if (parse_image(t, &im) != 0) { fprintf(stderr, "unknown image size '%s'\n", t); return -1; }
    if (c->nimages >= MAX_AXIS) return axis_overflow("images", t);
    c->images[c->nimages++] = im;
    return 0;
}
static int col_thread(const char *t, void *ud) {
    config_t *c = ud; int n = atoi(t);
    if (n <= 0) { fprintf(stderr, "bad thread count '%s'\n", t); return -1; }
    if (c->nthreads >= MAX_AXIS) return axis_overflow("threads", t);
    c->threads[c->nthreads++] = n;
    return 0;
}
static int col_model(const char *t, void *ud) {
    config_t *c = ud; model_t m;
    if (parse_model(t, &m) != 0) { fprintf(stderr, "unknown model '%s'\n", t); return -1; }
    if (c->nmodels >= MAX_AXIS) return axis_overflow("models", t);
    c->models[c->nmodels++] = m;
    return 0;
}
static int col_sched(const char *t, void *ud) {
    config_t *c = ud; conv_sched_t s;
    if (parse_sched(t, &s) != 0) { fprintf(stderr, "unknown schedule '%s'\n", t); return -1; }
    if (c->nscheds >= MAX_AXIS) return axis_overflow("schedule", t);
    c->scheds[c->nscheds++] = s;
    return 0;
}
static int col_chunk(const char *t, void *ud) {
    config_t *c = ud; int n = atoi(t);
    if (c->nchunks >= MAX_AXIS) return axis_overflow("chunks", t);
    c->chunks[c->nchunks++] = n;
    return 0;
}

static void usage(const char *prog)
{
    printf(
"Usage: %s [options]\n"
"  --patterns LIST    comma subset of sobel,gaussian,box (default all)\n"
"  --sizes LIST       kernel sizes, e.g. 3,7,11,15 (default 3,7)\n"
"  --images LIST      image sizes 2k,4k,8k (default 2k,4k,8k)\n"
"  --threads LIST     thread counts, e.g. 1,2,4,8 (default 1,2,4,8)\n"
"  --models LIST      seq,pthreads,omp-rows,omp-tiled (default all)\n"
"  --schedule LIST    omp-rows schedules: static,dynamic,guided (default static)\n"
"  --chunks LIST      omp-rows chunk sizes, e.g. 1,8,64,512 (default 64)\n"
"  --tile N           omp-tiled tile size (default 64)\n"
"  --reps N           timed repetitions (default 10)\n"
"  --warmups N        discarded warmup runs (default 2)\n"
"  --tdp-watts W      nominal package TDP for the energy ESTIMATE (default 65;\n"
"                     Ryzen 5 5600G catalog TDP; override for PPT/PBO values)\n"
"  --seed N           synthetic image seed (default 12345)\n"
"  --out PATH         CSV output path (default results/results.csv)\n"
"  --allow-oversubscribe  permit thread counts above available CPUs\n"
"  --quick            tiny images + reduced axes + few reps for a smoke run\n"
"  --help             show this help\n"
"\n"
"Notes: Sobel is only valid at kernel size 3; invalid (pattern,size) pairs are\n"
"skipped. Thread counts above the number of online CPUs are skipped unless\n"
"--allow-oversubscribe is given. On hosts without RAPL the energy column is a\n"
"TDP_ESTIMATE, never a direct measurement.\n", prog);
}

/* ------------------------------------------------------------------------- */
/* One benchmark cell                                                         */
/* ------------------------------------------------------------------------- */

/* Run the selected model once into `out`. */
static void run_model(model_t model, const image_t *in, const kernel_t *kern,
                      image_t *out, int threads, conv_sched_t sched,
                      int chunk, int tile)
{
    switch (model) {
    case MODEL_SEQ:
        convolve_sequential(in, kern, out);
        break;
    case MODEL_PTHREADS:
        convolve_pthreads(in, kern, out, threads);
        break;
    case MODEL_OMP_ROWS:
#ifdef _OPENMP
        omp_set_num_threads(threads);
#endif
        convolve_openmp_rows(in, kern, out, sched, chunk);
        break;
    case MODEL_OMP_TILED:
#ifdef _OPENMP
        omp_set_num_threads(threads);
#endif
        convolve_openmp_tiled(in, kern, out, tile);
        break;
    default:
        break;
    }
}

/* ------------------------------------------------------------------------- */
/* main                                                                       */
/* ------------------------------------------------------------------------- */

static void set_defaults(config_t *c)
{
    memset(c, 0, sizeof(*c));
    c->patterns[0] = PATTERN_SOBEL;
    c->patterns[1] = PATTERN_GAUSSIAN;
    c->patterns[2] = PATTERN_BOX;
    c->npatterns = 3;

    c->sizes[0] = 3; c->sizes[1] = 7; c->nsizes = 2;

    parse_image("2k", &c->images[0]);
    parse_image("4k", &c->images[1]);
    parse_image("8k", &c->images[2]);
    c->nimages = 3;

    c->threads[0] = 1; c->threads[1] = 2; c->threads[2] = 4; c->threads[3] = 8;
    c->nthreads = 4;

    c->models[0] = MODEL_SEQ; c->models[1] = MODEL_PTHREADS;
    c->models[2] = MODEL_OMP_ROWS; c->models[3] = MODEL_OMP_TILED;
    c->nmodels = 4;

    c->scheds[0] = CONV_SCHED_STATIC; c->nscheds = 1;
    c->chunks[0] = 64; c->nchunks = 1;
    c->tile = 64;
    c->reps = 10;
    c->warmups = 2;
    c->tdp_watts = 65.0;
    c->out_path = "results/results.csv";
    c->seed = 12345u;
    c->allow_oversubscribe = 0;
}

static void apply_quick(config_t *c)
{
    /* Tiny stand-in images, reduced axes, few reps -> fast smoke run. */
    c->images[0].name = "tiny"; c->images[0].w = 128; c->images[0].h = 96; c->images[0].ch = 3;
    c->images[1].name = "small"; c->images[1].w = 192; c->images[1].h = 128; c->images[1].ch = 1;
    c->nimages = 2;

    c->sizes[0] = 3; c->sizes[1] = 7; c->nsizes = 2;

    c->patterns[0] = PATTERN_SOBEL;
    c->patterns[1] = PATTERN_GAUSSIAN;
    c->patterns[2] = PATTERN_BOX;
    c->npatterns = 3;

    c->threads[0] = 1; c->threads[1] = 2; c->threads[2] = 4; c->threads[3] = 8;
    c->nthreads = 4;

    c->models[0] = MODEL_SEQ; c->models[1] = MODEL_PTHREADS;
    c->models[2] = MODEL_OMP_ROWS; c->models[3] = MODEL_OMP_TILED;
    c->nmodels = 4;

    c->scheds[0] = CONV_SCHED_STATIC; c->nscheds = 1;
    c->chunks[0] = 64; c->nchunks = 1;
    c->reps = 3;
    c->warmups = 1;
}

/* Ensure the parent directory of `path` exists (single level, best effort). */
static void ensure_parent_dir(const char *path)
{
    char buf[1024];
    strncpy(buf, path, sizeof(buf) - 1);
    buf[sizeof(buf) - 1] = '\0';
    char *slash = strrchr(buf, '/');
    if (!slash) return;
    *slash = '\0';
    if (buf[0] == '\0') return;
    if (plat_mkdir(buf) != 0) {
        fprintf(stderr, "warning: could not create dir '%s'\n", buf);
    }
}

int main(int argc, char **argv)
{
    config_t cfg;
    set_defaults(&cfg);

    /* The axes a user explicitly sets replace the defaults; we reset the
     * count to 0 the first time an axis flag is seen. */
    int seen_patterns = 0, seen_sizes = 0, seen_images = 0, seen_threads = 0;
    int seen_models = 0, seen_scheds = 0, seen_chunks = 0;
    int quick = 0;

    for (int i = 1; i < argc; ++i) {
        const char *a = argv[i];
        #define NEEDVAL() do { if (i + 1 >= argc) { \
            fprintf(stderr, "missing value for %s\n", a); return 2; } } while (0)

        if (strcmp(a, "--help") == 0 || strcmp(a, "-h") == 0) {
            usage(argv[0]); return 0;
        } else if (strcmp(a, "--quick") == 0) {
            quick = 1;
        } else if (strcmp(a, "--allow-oversubscribe") == 0) {
            cfg.allow_oversubscribe = 1;
        } else if (strcmp(a, "--patterns") == 0) {
            NEEDVAL(); if (!seen_patterns) { cfg.npatterns = 0; seen_patterns = 1; }
            if (for_each_token(argv[++i], col_pattern, &cfg) != 0) return 2;
        } else if (strcmp(a, "--sizes") == 0) {
            NEEDVAL(); if (!seen_sizes) { cfg.nsizes = 0; seen_sizes = 1; }
            if (for_each_token(argv[++i], col_size, &cfg) != 0) return 2;
        } else if (strcmp(a, "--images") == 0) {
            NEEDVAL(); if (!seen_images) { cfg.nimages = 0; seen_images = 1; }
            if (for_each_token(argv[++i], col_image, &cfg) != 0) return 2;
        } else if (strcmp(a, "--threads") == 0) {
            NEEDVAL(); if (!seen_threads) { cfg.nthreads = 0; seen_threads = 1; }
            if (for_each_token(argv[++i], col_thread, &cfg) != 0) return 2;
        } else if (strcmp(a, "--models") == 0) {
            NEEDVAL(); if (!seen_models) { cfg.nmodels = 0; seen_models = 1; }
            if (for_each_token(argv[++i], col_model, &cfg) != 0) return 2;
        } else if (strcmp(a, "--schedule") == 0) {
            NEEDVAL(); if (!seen_scheds) { cfg.nscheds = 0; seen_scheds = 1; }
            if (for_each_token(argv[++i], col_sched, &cfg) != 0) return 2;
        } else if (strcmp(a, "--chunks") == 0) {
            NEEDVAL(); if (!seen_chunks) { cfg.nchunks = 0; seen_chunks = 1; }
            if (for_each_token(argv[++i], col_chunk, &cfg) != 0) return 2;
        } else if (strcmp(a, "--tile") == 0) {
            NEEDVAL(); cfg.tile = atoi(argv[++i]);
        } else if (strcmp(a, "--reps") == 0) {
            NEEDVAL(); cfg.reps = atoi(argv[++i]);
        } else if (strcmp(a, "--warmups") == 0) {
            NEEDVAL(); cfg.warmups = atoi(argv[++i]);
        } else if (strcmp(a, "--tdp-watts") == 0) {
            NEEDVAL(); cfg.tdp_watts = atof(argv[++i]);
        } else if (strcmp(a, "--seed") == 0) {
            NEEDVAL(); cfg.seed = (uint32_t)strtoul(argv[++i], NULL, 10);
        } else if (strcmp(a, "--out") == 0) {
            NEEDVAL(); cfg.out_path = argv[++i];
        } else {
            fprintf(stderr, "unknown option '%s' (try --help)\n", a);
            return 2;
        }
        #undef NEEDVAL
    }

    if (quick) apply_quick(&cfg);

    if (cfg.reps < 1) cfg.reps = 1;
    if (cfg.warmups < 0) cfg.warmups = 0;
    if (cfg.tile < 1) cfg.tile = 64;

    long ncpu = plat_online_cpus();
    if (ncpu < 1) ncpu = 1;

    /* ---- Open CSV --------------------------------------------------------- */
    ensure_parent_dir(cfg.out_path);
    FILE *csv = fopen(cfg.out_path, "w");
    if (!csv) {
        fprintf(stderr, "error: cannot open CSV '%s': %s\n",
                cfg.out_path, strerror(errno));
        return 1;
    }
    fputs(CSV_HEADER, csv);

    /* ---- Banner ----------------------------------------------------------- */
    printf("== 2D Convolution Benchmark ==\n");
    printf("online CPUs: %ld | reps: %d | warmups: %d | tdp-watts: %.1f\n",
           ncpu, cfg.reps, cfg.warmups, cfg.tdp_watts);
#ifdef _OPENMP
    printf("OpenMP: enabled (omp_get_num_procs=%d)\n", omp_get_num_procs());
#else
    printf("OpenMP: DISABLED at compile time (omp models run sequentially)\n");
#endif
    printf("CSV: %s\n", cfg.out_path);
    printf("Energy backend: %s\n", energy_probe());
    printf("Energy note: RAPL used when present; otherwise a TDP_ESTIMATE "
           "(not a direct measurement).\n");
    printf("CAVEATS (read before interpreting the numbers below):\n");
    printf("  * MFLOP/W under TDP_ESTIMATE is NOT comparable across models. In\n"
           "    that fallback avg_power_W reduces to tdp_watts*(busy_cores/ncpu),\n"
           "    i.e. power tracks core occupancy, not a measured energy\n"
           "    difference; use MFLOP/W only to compare runs of the SAME model.\n");
    printf("  * 'sobel' is the linear horizontal Gx 3x3 kernel, NOT gradient-\n"
           "    magnitude edge detection (that needs two passes); it is used so\n"
           "    all four implementations stay bit-for-bit comparable.\n");
    printf("  * FLOPs/arith_int are a NOMINAL dense-kernel upper bound (2*k*k per\n"
           "    output sample); they do not subtract Sobel's zero column or the\n"
           "    skipped zero-padded border taps, so absolute FLOP-derived values\n"
           "    are slightly overstated (identically across models, so speedup\n"
           "    and efficiency ratios are unaffected).\n\n");

    /* ---- Table header ----------------------------------------------------- */
    printf("%-9s %3s %-6s %-9s %3s %-7s %5s | %10s %7s %6s | "
           "%8s %7s %10s %-12s %5s\n",
           "pattern", "k", "image", "model", "thr", "sched", "chunk",
           "time_ms", "speedup", "eff", "power_W", "MFLOP/W", "arith_int",
           "energy_src", "valid");
    printf("------------------------------------------------------------"
           "------------------------------------------------------------\n");

    int total_rows = 0, failed_rows = 0;

    /* ---- Sweep ------------------------------------------------------------ */
    for (int pi = 0; pi < cfg.npatterns; ++pi) {
        conv_pattern_t pattern = cfg.patterns[pi];

        for (int si = 0; si < cfg.nsizes; ++si) {
            int k = cfg.sizes[si];

            /* Skip invalid (pattern,size) pairs (e.g. sobel at k != 3). */
            kernel_t kern;
            if (kernel_build(pattern, k, &kern) != 0) {
                continue;
            }

            for (int ii = 0; ii < cfg.nimages; ++ii) {
                imgsize_t im = cfg.images[ii];

                /* Generate the image ONCE (outside timing). */
                image_t in;
                if (image_generate(&in, im.w, im.h, im.ch, cfg.seed) != 0) {
                    fprintf(stderr, "error: image_generate failed for %s\n", im.name);
                    kernel_free(&kern);
                    goto cleanup_fail;
                }

                /* Sequential reference ONCE (outside timing). */
                image_t ref;
                if (image_alloc(&ref, im.w, im.h, im.ch) != 0) {
                    image_free(&in); kernel_free(&kern); goto cleanup_fail;
                }
                convolve_sequential(&in, &kern, &ref);

                double flops = metrics_flops(im.w, im.h, im.ch, k);
                double bytes = metrics_bytes_moved(im.w, im.h, im.ch, k);
                double ai = metrics_arithmetic_intensity(flops, bytes);

                /* Sequential baseline time for speedup/efficiency. Timed here
                 * ONCE per (pattern,size,image) so speedup is always relative
                 * to the sequential reference even if --models omits seq. The
                 * baseline uses warmups + reps and times only the convolve. */
                double t_seq_mean = 0.0;
                {
                    image_t btmp;
                    if (image_alloc(&btmp, im.w, im.h, im.ch) != 0) {
                        image_free(&in); image_free(&ref);
                        kernel_free(&kern); goto cleanup_fail;
                    }
                    for (int w = 0; w < cfg.warmups; ++w)
                        convolve_sequential(&in, &kern, &btmp);
                    double bsum = 0.0;
                    for (int r = 0; r < cfg.reps; ++r) {
                        double t0 = metrics_now_seconds();
                        convolve_sequential(&in, &kern, &btmp);
                        double t1 = metrics_now_seconds();
                        bsum += (t1 - t0);
                    }
                    image_free(&btmp);
                    t_seq_mean = bsum / (double)cfg.reps;
                }

                for (int mi = 0; mi < cfg.nmodels; ++mi) {
                    model_t model = cfg.models[mi];

                    /* Determine the inner loops for this model. */
                    int seq_only = (model == MODEL_SEQ);

                    for (int ti = 0; ti < cfg.nthreads; ++ti) {
                        int threads = seq_only ? 1 : cfg.threads[ti];

                        if (!seq_only && threads > ncpu && !cfg.allow_oversubscribe) {
                            /* Skip oversubscription unless explicitly allowed. */
                            continue;
                        }

                        /* schedule/chunk only vary for omp-rows. */
                        int sched_lo = 0, sched_hi = 1;
                        int chunk_lo = 0, chunk_hi = 1;
                        if (model == MODEL_OMP_ROWS) {
                            sched_hi = cfg.nscheds;
                            chunk_hi = cfg.nchunks;
                        }

                        for (int sc = sched_lo; sc < sched_hi; ++sc) {
                            conv_sched_t sched = (model == MODEL_OMP_ROWS)
                                ? cfg.scheds[sc] : CONV_SCHED_STATIC;

                            for (int ch = chunk_lo; ch < chunk_hi; ++ch) {
                                int chunk = (model == MODEL_OMP_ROWS)
                                    ? cfg.chunks[ch] : 0;

                                image_t out;
                                if (image_alloc(&out, im.w, im.h, im.ch) != 0) {
                                    image_free(&in); image_free(&ref);
                                    kernel_free(&kern); goto cleanup_fail;
                                }

                                /* Warmups (discarded). */
                                for (int w = 0; w < cfg.warmups; ++w) {
                                    run_model(model, &in, &kern, &out,
                                              threads, sched, chunk, cfg.tile);
                                }

                                /* Timed reps + energy over the whole timed
                                 * window (bracketing only convolve calls). */
                                double *times = malloc((size_t)cfg.reps * sizeof(double));
                                if (!times) {
                                    image_free(&out); image_free(&in);
                                    image_free(&ref); kernel_free(&kern);
                                    goto cleanup_fail;
                                }

                                energy_sample_t es0, es1;
                                double wall0 = metrics_now_seconds();
                                energy_begin(&es0);
                                for (int r = 0; r < cfg.reps; ++r) {
                                    double t0 = metrics_now_seconds();
                                    run_model(model, &in, &kern, &out,
                                              threads, sched, chunk, cfg.tile);
                                    double t1 = metrics_now_seconds();
                                    times[r] = t1 - t0;
                                }
                                energy_end(&es1);
                                double wall1 = metrics_now_seconds();

                                /* Validate the LAST output vs the reference. */
                                validate_result_t vr = validate(&ref, &out, VALIDATE_EPS);

                                stats_t st = metrics_stats(times, (size_t)cfg.reps);
                                free(times);
                                image_free(&out);

                                /* Energy over the whole timed window. */
                                double window = wall1 - wall0;
                                energy_result_t er = energy_compute(&es0, &es1,
                                        window, cfg.tdp_watts);
                                /* Per-rep energy / power for this config. */
                                double per_rep_energy = er.energy_j / (double)cfg.reps;
                                double avg_power = er.avg_power_w; /* W over window */
                                double edp = metrics_edp(per_rep_energy, st.mean);
                                double mflop_w = metrics_mflop_per_watt(flops, avg_power);

                                double speedup = (t_seq_mean > 0.0)
                                    ? metrics_speedup(t_seq_mean, st.mean) : 0.0;
                                double eff = metrics_efficiency(speedup, threads);
                                double overhead = (threads == 1 && t_seq_mean > 0.0)
                                    ? metrics_overhead_pct(t_seq_mean, st.mean) : 0.0;

                                const char *valid = vr.pass ? "PASS" : "FAIL";
                                total_rows++;
                                if (!vr.pass) failed_rows++;

                                /* CSV row (metrics still recorded; validation
                                 * column marks FAIL so invalid rows are
                                 * identifiable and not trusted). */
                                fprintf(csv,
                                    "%s,%d,%s,%d,%d,%d,%s,%d,%s,%d,%d,"
                                    "%.9g,%.9g,%.9g,%.6g,%.6g,%.6g,%.4g,"
                                    "%.6g,%.6g,%.6g,%.6g,%.6g,%.6g,%.6g,%.4g,%s,%s\n",
                                    kernel_pattern_name(pattern), k, im.name,
                                    im.w, im.h, im.ch, model_name(model),
                                    threads,
                                    (model == MODEL_OMP_ROWS) ? sched_name(sched) : "-",
                                    (model == MODEL_OMP_ROWS) ? chunk : 0,
                                    cfg.reps,
                                    st.mean, st.median, st.stddev, st.cv,
                                    speedup, eff, overhead,
                                    flops, bytes, ai,
                                    per_rep_energy, avg_power, edp, mflop_w,
                                    er.cpu_utilization,
                                    energy_source_label(er.source), valid);

                                /* Only print valid rows in the table (flag
                                 * invalid ones distinctly). */
                                printf("%-9s %3d %-6s %-9s %3d %-7s %5d | "
                                       "%10.4f %7.3f %6.3f | %8.2f %7.2f "
                                       "%10.3f %-12s %5s\n",
                                       kernel_pattern_name(pattern), k, im.name,
                                       model_name(model), threads,
                                       (model == MODEL_OMP_ROWS) ? sched_name(sched) : "-",
                                       (model == MODEL_OMP_ROWS) ? chunk : 0,
                                       st.mean * 1000.0, speedup, eff,
                                       avg_power, mflop_w, ai,
                                       energy_source_label(er.source), valid);
                            } /* chunk */
                        } /* sched */

                        if (seq_only) break; /* seq has no thread axis */
                    } /* threads */
                } /* models */

                image_free(&ref);
                image_free(&in);
            } /* images */

            kernel_free(&kern);
        } /* sizes */
    } /* patterns */

    fclose(csv);

    printf("\nDone. %d configurations recorded (%d FAILED validation).\n",
           total_rows, failed_rows);
    printf("CSV written to %s\n", cfg.out_path);
    if (failed_rows > 0) {
        fprintf(stderr, "WARNING: %d configuration(s) failed pixel-by-pixel "
                "validation; their metrics are flagged FAIL and must not be "
                "trusted.\n", failed_rows);
        return 1;
    }
    return 0;

cleanup_fail:
    fclose(csv);
    fprintf(stderr, "error: aborted due to allocation/generation failure\n");
    return 1;
}


Writing /content/Artigo_PCD-main/src/main.c


In [14]:
%%writefile /content/Artigo_PCD-main/src/metrics.c
/*
 * metrics.c - Implementation of timing, FLOP/byte models, statistics and the
 * derived performance/energy metrics (including MFLOP/W). See metrics.h.
 *
 * The monotonic clock is obtained through the portable platform layer so this
 * TU no longer calls any POSIX API directly (qsort/math only).
 */
#include "metrics.h"
#include "platform.h"

#include <math.h>
#include <stdlib.h>
#include <string.h>

double metrics_now_seconds(void)
{
    return plat_monotonic_seconds();
}

double metrics_flops(int out_w, int out_h, int channels, int k)
{
    return (double)out_w * (double)out_h * (double)channels
           * 2.0 * (double)k * (double)k;
}

double metrics_bytes_moved(int out_w, int out_h, int channels, int k)
{
    return (double)out_w * (double)out_h * (double)channels
           * ((double)k * (double)k + 1.0) * 4.0;
}

double metrics_arithmetic_intensity(double flops, double bytes_moved)
{
    if (bytes_moved <= 0.0) return 0.0;
    return flops / bytes_moved;
}

static int cmp_double(const void *a, const void *b)
{
    double x = *(const double *)a;
    double y = *(const double *)b;
    if (x < y) return -1;
    if (x > y) return 1;
    return 0;
}

stats_t metrics_stats(const double *samples, size_t n)
{
    stats_t s;
    memset(&s, 0, sizeof(s));
    if (n == 0) return s;
    s.n = n;

    double *tmp = (double *)malloc(n * sizeof(double));
    if (!tmp) {
        /* Degrade gracefully: compute mean only, no median/stddev. */
        double sum = 0.0;
        for (size_t i = 0; i < n; ++i) sum += samples[i];
        s.mean = sum / (double)n;
        s.median = s.mean;
        s.min = s.max = s.mean;
        return s;
    }
    memcpy(tmp, samples, n * sizeof(double));
    qsort(tmp, n, sizeof(double), cmp_double);

    double sum = 0.0;
    for (size_t i = 0; i < n; ++i) sum += tmp[i];
    s.mean = sum / (double)n;
    s.min = tmp[0];
    s.max = tmp[n - 1];

    if (n % 2 == 1) {
        s.median = tmp[n / 2];
    } else {
        s.median = 0.5 * (tmp[n / 2 - 1] + tmp[n / 2]);
    }

    if (n > 1) {
        double acc = 0.0;
        for (size_t i = 0; i < n; ++i) {
            double d = tmp[i] - s.mean;
            acc += d * d;
        }
        s.stddev = sqrt(acc / (double)(n - 1));
    } else {
        s.stddev = 0.0;
    }
    s.cv = (s.mean != 0.0) ? (s.stddev / s.mean) : 0.0;

    free(tmp);
    return s;
}

double metrics_speedup(double t_seq, double t_par)
{
    if (t_par <= 0.0) return 0.0;
    return t_seq / t_par;
}

double metrics_efficiency(double speedup, int threads)
{
    if (threads <= 0) return 0.0;
    return speedup / (double)threads;
}

double metrics_overhead_pct(double t_seq, double t_par1)
{
    if (t_seq <= 0.0) return 0.0;
    return 100.0 * (t_par1 - t_seq) / t_seq;
}

double metrics_edp(double energy_j, double time_s)
{
    return energy_j * time_s;
}

double metrics_mflop_per_watt(double flops, double avg_power_w)
{
    if (avg_power_w <= 0.0) return 0.0;
    return (flops / 1e6) / avg_power_w;
}


Writing /content/Artigo_PCD-main/src/metrics.c


In [15]:
%%writefile /content/Artigo_PCD-main/src/metrics.h
/*
 * metrics.h - Timing, FLOP model, statistics and derived performance/energy
 * metrics for the convolution benchmark.
 *
 * Timing covers ONLY the convolution kernel. The harness brackets each
 * convolve call with metrics_now_seconds() (CLOCK_MONOTONIC) so image
 * generation and file I/O are excluded.
 *
 * FLOP model (agreed project-wide):
 *   flops = (double)out_w * out_h * channels * 2.0 * k * k
 * i.e. k*k multiplies + k*k adds per output sample.
 *
 * Bytes-moved model (for arithmetic intensity): the dominant traffic per
 * output sample is reading the k*k input window (4 bytes/float) plus writing
 * the single output sample (4 bytes). We charge:
 *   bytes = (double)out_w * out_h * channels * (k*k + 1) * 4.0
 * This is a simple, documented upper-bound-ish model (ignores cache reuse),
 * so arithmetic intensity grows ~ with k*k, matching the Roofline sweep from
 * memory-bound (small k) to compute-bound (large k).
 */
#ifndef METRICS_H
#define METRICS_H

#include <stddef.h>

/* Monotonic wall-clock reading in seconds (CLOCK_MONOTONIC). */
double metrics_now_seconds(void);

/* flops = out_w*out_h*channels*2*k*k */
double metrics_flops(int out_w, int out_h, int channels, int k);

/* bytes = out_w*out_h*channels*(k*k+1)*4 */
double metrics_bytes_moved(int out_w, int out_h, int channels, int k);

/* arithmetic intensity (FLOP/byte) */
double metrics_arithmetic_intensity(double flops, double bytes_moved);

/* Aggregate statistics over a set of repetition times (seconds). */
typedef struct {
    double mean;
    double median;
    double stddev;   /* sample standard deviation (n-1) */
    double cv;       /* coefficient of variation = stddev/mean */
    double min;
    double max;
    size_t n;
} stats_t;

/*
 * Compute stats over `n` samples. The input array is not modified (a local
 * copy is sorted for the median). Returns zeroed stats when n == 0.
 */
stats_t metrics_stats(const double *samples, size_t n);

/* Speedup S(p) = t_seq / t_par. */
double metrics_speedup(double t_seq, double t_par);

/* Parallel efficiency E(p) = S(p) / p. */
double metrics_efficiency(double speedup, int threads);

/* Overhead of the 1-thread parallel run vs sequential, as a percentage:
 *   100 * (t_par1 - t_seq) / t_seq */
double metrics_overhead_pct(double t_seq, double t_par1);

/* Energy-delay product = energy_j * time_s. */
double metrics_edp(double energy_j, double time_s);

/* The user-requested metric: MFLOP/W = (flops/1e6) / avg_power_w. */
double metrics_mflop_per_watt(double flops, double avg_power_w);

#endif /* METRICS_H */


Writing /content/Artigo_PCD-main/src/metrics.h


In [16]:
%%writefile /content/Artigo_PCD-main/src/platform.c
/*
 * platform.c - Implementation of the thin cross-platform compatibility layer.
 *
 * All platform branching for the engine's POSIX dependencies lives here,
 * selected by a single `#if defined(_WIN32)` split:
 *
 *   POSIX branch   : pthreads, clock_gettime(CLOCK_MONOTONIC),
 *                    clock_gettime(CLOCK_PROCESS_CPUTIME_ID),
 *                    sysconf(_SC_NPROCESSORS_ONLN), mkdir(path,0777).
 *   WINDOWS branch : CreateThread + WaitForSingleObject/CloseHandle,
 *                    QueryPerformanceCounter/QueryPerformanceFrequency,
 *                    GetProcessTimes, GetActiveProcessorCount/GetSystemInfo,
 *                    _mkdir.
 *
 * The _POSIX_C_SOURCE define below (needed by glibc for clock_gettime and the
 * CLOCK_* macros) is harmless on Windows because the POSIX branch is compiled
 * out entirely there.
 */
#if !defined(_WIN32)
#define _POSIX_C_SOURCE 200809L
#endif

#include "platform.h"

#if defined(_WIN32)
/* ------------------------------------------------------------- Windows ---- */

#include <windows.h>
#include <direct.h>
#include <errno.h>
#include <stdlib.h>  /* malloc/free used by the thread trampoline below */

/*
 * Trampoline so the engine can keep the pthread-style worker signature
 * `void *(*)(void *)` while Win32 requires a `DWORD WINAPI (*)(LPVOID)`
 * thread procedure. We heap-allocate the bookkeeping so the thread owns it
 * and frees it on exit.
 */
typedef struct {
    void *(*fn)(void *);
    void *arg;
} plat_tramp_t;

static DWORD WINAPI plat_thread_trampoline(LPVOID p)
{
    plat_tramp_t *tr = (plat_tramp_t *)p;
    void *(*fn)(void *) = tr->fn;
    void *arg = tr->arg;
    free(tr);
    fn(arg);
    return 0;
}

int plat_thread_create(plat_thread_t *t, void *(*fn)(void *), void *arg)
{
    if (!t || !fn) return -1;
    plat_tramp_t *tr = (plat_tramp_t *)malloc(sizeof(*tr));
    if (!tr) return -1;
    tr->fn = fn;
    tr->arg = arg;
    HANDLE h = CreateThread(NULL, 0, plat_thread_trampoline, tr, 0, NULL);
    if (h == NULL) {
        free(tr);
        return -1;
    }
    *t = (plat_thread_t)h;
    return 0;
}

int plat_thread_join(plat_thread_t t)
{
    HANDLE h = (HANDLE)t;
    if (h == NULL) return -1;
    if (WaitForSingleObject(h, INFINITE) != WAIT_OBJECT_0) {
        CloseHandle(h);
        return -1;
    }
    CloseHandle(h);
    return 0;
}

double plat_monotonic_seconds(void)
{
    LARGE_INTEGER freq, now;
    if (!QueryPerformanceFrequency(&freq) || freq.QuadPart == 0)
        return 0.0;
    QueryPerformanceCounter(&now);
    return (double)now.QuadPart / (double)freq.QuadPart;
}

uint64_t plat_process_cpu_time_ns(void)
{
    FILETIME creation, exit, kernel, user;
    if (!GetProcessTimes(GetCurrentProcess(), &creation, &exit,
                         &kernel, &user)) {
        return 0;
    }
    /* kernel/user are in 100-nanosecond units. */
    ULARGE_INTEGER k, u;
    k.LowPart = kernel.dwLowDateTime;
    k.HighPart = kernel.dwHighDateTime;
    u.LowPart = user.dwLowDateTime;
    u.HighPart = user.dwHighDateTime;
    uint64_t ticks100ns = (uint64_t)k.QuadPart + (uint64_t)u.QuadPart;
    return ticks100ns * 100ull;
}

long plat_online_cpus(void)
{
    DWORD n = GetActiveProcessorCount(ALL_PROCESSOR_GROUPS);
    if (n == 0) {
        SYSTEM_INFO si;
        GetSystemInfo(&si);
        n = si.dwNumberOfProcessors;
    }
    return (n >= 1) ? (long)n : 1L;
}

int plat_mkdir(const char *path)
{
    if (_mkdir(path) == 0) return 0;
    if (errno == EEXIST) return 0;
    return -1;
}

#else
/* --------------------------------------------------------------- POSIX ---- */

#include <errno.h>
#include <pthread.h>
#include <sys/stat.h>
#include <sys/types.h>
#include <time.h>
#include <unistd.h>

int plat_thread_create(plat_thread_t *t, void *(*fn)(void *), void *arg)
{
    return pthread_create(t, NULL, fn, arg);
}

int plat_thread_join(plat_thread_t t)
{
    return pthread_join(t, NULL);
}

double plat_monotonic_seconds(void)
{
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return (double)ts.tv_sec + (double)ts.tv_nsec * 1e-9;
}

uint64_t plat_process_cpu_time_ns(void)
{
    struct timespec ts;
    if (clock_gettime(CLOCK_PROCESS_CPUTIME_ID, &ts) == 0) {
        return (uint64_t)ts.tv_sec * 1000000000ull + (uint64_t)ts.tv_nsec;
    }
    return 0;
}

long plat_online_cpus(void)
{
    long n = sysconf(_SC_NPROCESSORS_ONLN);
    return (n >= 1) ? n : 1L;
}

int plat_mkdir(const char *path)
{
    if (mkdir(path, 0777) == 0) return 0;
    if (errno == EEXIST) return 0;
    return -1;
}

#endif /* _WIN32 */


Writing /content/Artigo_PCD-main/src/platform.c


In [17]:
%%writefile /content/Artigo_PCD-main/src/platform.h
/*
 * platform.h - Thin cross-platform compatibility layer.
 *
 * This header declares a minimal portable API that encapsulates every
 * Linux/POSIX dependency used by the benchmark engine (threads, monotonic
 * clock, process CPU time, online CPU count, directory creation). The engine
 * translation units (convolution.c, metrics.c, energy.c, main.c, selftest.c)
 * call ONLY these plat_* functions; all platform branching lives inside
 * platform.c behind a single `#if defined(_WIN32)` split.
 *
 * On POSIX the implementation uses pthreads, clock_gettime, sysconf and
 * mkdir, so the Linux behavior is byte-for-byte identical to before this
 * layer existed. On Windows the implementation uses the Win32 API
 * (CreateThread, QueryPerformanceCounter, GetProcessTimes,
 * GetActiveProcessorCount, _mkdir). RAPL energy sampling is Linux-only and
 * stays inside energy.c under its own guard; it is not part of this API.
 */
#ifndef PLATFORM_H
#define PLATFORM_H

#include <stdint.h>

#ifdef __cplusplus
extern "C" {
#endif

/*
 * Opaque thread handle. The underlying type differs per platform but the
 * engine only ever passes it by value to plat_thread_join, so an opaque
 * typedef keeps the API stable. On POSIX it wraps pthread_t; on Windows it
 * wraps a HANDLE (plus the trampoline bookkeeping is internal to platform.c).
 */
#if defined(_WIN32)
typedef void *plat_thread_t; /* Win32 HANDLE */
#else
#include <pthread.h>
typedef pthread_t plat_thread_t;
#endif

/*
 * Create a thread running fn(arg). The worker keeps the pthread-style
 * signature `void *(*)(void *)` on every platform (on Windows a tiny
 * trampoline adapts it to the Win32 thread-proc). Returns 0 on success and
 * non-zero on failure (mirroring pthread_create).
 */
int plat_thread_create(plat_thread_t *t, void *(*fn)(void *), void *arg);

/* Join a thread previously created with plat_thread_create. Returns 0 on
 * success, non-zero on failure. */
int plat_thread_join(plat_thread_t t);

/* Monotonic wall-clock time in seconds (CLOCK_MONOTONIC on POSIX,
 * QueryPerformanceCounter on Windows). */
double plat_monotonic_seconds(void);

/* Process CPU time (user + system) in nanoseconds. */
uint64_t plat_process_cpu_time_ns(void);

/* Number of online/active CPUs (always >= 1). */
long plat_online_cpus(void);

/* Create a directory. Returns 0 on success or if it already exists,
 * non-zero on genuine failure. */
int plat_mkdir(const char *path);

#ifdef __cplusplus
}
#endif

#endif /* PLATFORM_H */


Writing /content/Artigo_PCD-main/src/platform.h


In [18]:
%%writefile /content/Artigo_PCD-main/src/selftest.c
/*
 * selftest.c - Minimal smoke driver for FEAT-001.
 *
 * Proves that the four convolution implementations agree pixel-by-pixel on a
 * small synthetic image, and that synthetic image generation + binary
 * PGM/PPM round-trip work. This is a throwaway driver; the real benchmark
 * harness arrives in a later feature. It is only compiled when SELFTEST is
 * defined.
 *
 * Build (from /projects/sandbox): see the feature verification steps. The
 * parallel paths are compiled WITH -fopenmp -pthread; the sequential
 * reference is verified to compile WITHOUT -fopenmp (the _OPENMP guard in
 * convolution.c makes the OpenMP entry points fall back to sequential when
 * the flag is absent).
 */
#ifdef SELFTEST

#include "convolution.h"
#include "image.h"
#include "kernels.h"

#include <stdio.h>
#include <stdlib.h>

static int check_variant(const char *label, const image_t *ref,
                         const image_t *got, double eps)
{
    validate_result_t v = validate(ref, got, eps);
    if (v.pass) {
        printf("  [PASS] %-14s vs sequential  (max abs diff = %.3g)\n",
               label, v.max_abs_diff);
        return 0;
    }
    printf("  [FAIL] %-14s vs sequential  (max abs diff = %.6g, "
           "first mismatch at x=%d y=%d c=%d)\n",
           label, v.max_abs_diff, v.mm_x, v.mm_y, v.mm_c);
    return 1;
}

/* Round-trip test for PGM/PPM I/O. Returns 0 on success. */
static int test_io_roundtrip(void)
{
    int fails = 0;

    /* PGM (grayscale). */
    image_t g = {0};
    if (image_generate(&g, 40, 30, 1, 1234u) != 0) {
        printf("  [FAIL] generate PGM image\n");
        return 1;
    }
    if (image_write_pgm(&g, "build/selftest.pgm") != 0) {
        printf("  [FAIL] write PGM\n");
        image_free(&g);
        return 1;
    }
    image_t g2 = {0};
    if (image_read_pgm(&g2, "build/selftest.pgm") != 0) {
        printf("  [FAIL] read PGM\n");
        image_free(&g);
        return 1;
    }
    /* PGM/PPM store 8-bit samples, so a float image round-trips to the
     * nearest integer. The inherent quantization error is at most half a
     * level; use 0.5001 as the tolerance (not 0 -- that would wrongly treat
     * lossless 8-bit storage as a failure). */
    const double io_eps = 0.5001;
    validate_result_t vg = validate(&g, &g2, io_eps);
    if (vg.pass)
        printf("  [PASS] PGM round-trip  (%dx%d, max abs diff = %.3g)\n",
               g.width, g.height, vg.max_abs_diff);
    else {
        printf("  [FAIL] PGM round-trip  (max abs diff = %.6g)\n",
               vg.max_abs_diff);
        ++fails;
    }
    /* Second write/read of the already-8-bit-quantized buffer must be
     * EXACTLY identical (lossless once values are integral). */
    if (image_write_pgm(&g2, "build/selftest2.pgm") == 0) {
        image_t g3 = {0};
        if (image_read_pgm(&g3, "build/selftest2.pgm") == 0) {
            validate_result_t ve = validate(&g2, &g3, 0.0);
            if (ve.pass)
                printf("  [PASS] PGM exact re-round-trip (quantized buffer "
                       "identical)\n");
            else {
                printf("  [FAIL] PGM exact re-round-trip (max abs diff = "
                       "%.6g)\n", ve.max_abs_diff);
                ++fails;
            }
            image_free(&g3);
        }
    }
    image_free(&g);
    image_free(&g2);

    /* PPM (color). */
    image_t c = {0};
    if (image_generate(&c, 24, 24, 3, 777u) != 0) {
        printf("  [FAIL] generate PPM image\n");
        return 1;
    }
    if (image_write_ppm(&c, "build/selftest.ppm") != 0) {
        printf("  [FAIL] write PPM\n");
        image_free(&c);
        return 1;
    }
    image_t c2 = {0};
    if (image_read_ppm(&c2, "build/selftest.ppm") != 0) {
        printf("  [FAIL] read PPM\n");
        image_free(&c);
        return 1;
    }
    validate_result_t vc = validate(&c, &c2, io_eps);
    if (vc.pass)
        printf("  [PASS] PPM round-trip  (%dx%d, max abs diff = %.3g)\n",
               c.width, c.height, vc.max_abs_diff);
    else {
        printf("  [FAIL] PPM round-trip  (max abs diff = %.6g)\n",
               vc.max_abs_diff);
        ++fails;
    }
    image_free(&c);
    image_free(&c2);

    return fails;
}

int main(void)
{
    const double eps = 1e-3; /* float conv order can differ slightly; tiny eps */
    int total_fail = 0;

    printf("=== FEAT-001 self-test ===\n");

    printf("-- image I/O --\n");
    total_fail += test_io_roundtrip();

    /* Small synthetic image with real edges (checkerboard + gradient). */
    image_t in = {0};
    if (image_generate(&in, 97, 71, 1, 42u) != 0) {
        printf("FATAL: image_generate failed\n");
        return 2;
    }

    /* Exercise every pattern/size combination for correctness. */
    struct { conv_pattern_t p; int k; } combos[] = {
        { PATTERN_SOBEL,    3 },
        { PATTERN_GAUSSIAN, 3 }, { PATTERN_GAUSSIAN, 7 },
        { PATTERN_GAUSSIAN, 11 }, { PATTERN_GAUSSIAN, 15 },
        { PATTERN_BOX,      3 }, { PATTERN_BOX, 7 },
        { PATTERN_BOX,     11 }, { PATTERN_BOX, 15 },
    };
    const int ncombo = (int)(sizeof(combos) / sizeof(combos[0]));

    for (int i = 0; i < ncombo; ++i) {
        kernel_t kern = {0};
        if (kernel_build(combos[i].p, combos[i].k, &kern) != 0) {
            printf("FATAL: kernel_build(%s, %d) failed\n",
                   kernel_pattern_name(combos[i].p), combos[i].k);
            total_fail++;
            continue;
        }

        image_t ref = {0}, pth = {0}, orow = {0}, otile = {0};
        image_alloc(&ref,   in.width, in.height, in.channels);
        image_alloc(&pth,   in.width, in.height, in.channels);
        image_alloc(&orow,  in.width, in.height, in.channels);
        image_alloc(&otile, in.width, in.height, in.channels);

        convolve_sequential(&in, &kern, &ref);
        convolve_pthreads(&in, &kern, &pth, 4);
        convolve_openmp_rows(&in, &kern, &orow, CONV_SCHED_DYNAMIC, 16);
        convolve_openmp_tiled(&in, &kern, &otile, 16);

        printf("-- %s --\n", kern.name);
        total_fail += check_variant("pthreads",    &ref, &pth,   eps);
        total_fail += check_variant("openmp-rows",  &ref, &orow,  eps);
        total_fail += check_variant("openmp-tiled", &ref, &otile, eps);

        image_free(&ref);
        image_free(&pth);
        image_free(&orow);
        image_free(&otile);
        kernel_free(&kern);
    }

    image_free(&in);

    printf("=== %s ===\n",
           total_fail == 0 ? "VALIDATION PASS" : "VALIDATION FAIL");
    return total_fail == 0 ? 0 : 1;
}

#endif /* SELFTEST */


Writing /content/Artigo_PCD-main/src/selftest.c


## Compilar, rodar e baixar o CSV

In [20]:
import os, csv, pandas as pd
from google.colab import files

# ---------- CONFIGURAÇÃO ----------
IMAGES   = "2k,4k,8k"    # tamanho imagem
SIZES    = "3,7,11,15"   # tamanhos de kernel
PATTERNS = "sobel,gaussian,box"
THREADS  = "1,2"         # o Colab tem ~2 vCPUs
REPS     = 10             # 10 = mais preciso
TDP      = 65            # W nominais, usado só na estimativa de energia
# -------------------------------------------------

os.chdir("/content/Artigo_PCD-main")
!pkill conv_bench || true
!apt-get -qq install -y build-essential > /dev/null
!make clean > /dev/null && make 2>&1 | tail -2

print("\n--- Ambiente e fonte de energia ---")
!nproc
!lscpu | grep -E "Model name"
!./build/conv_bench --quick --out results/probe.csv | grep -E "Energy backend|online CPUs"

print("\n--- Validação do código ---")
!make selftest 2>&1 | tail -2

print("\n--- Matriz completa---")
OUT = "/content/Artigo_PCD-main/results/results_colab.csv"
cmd = (f"./build/conv_bench --patterns {PATTERNS} --sizes {SIZES} --images {IMAGES} "
       f"--threads {THREADS} --models seq,pthreads,omp-rows,omp-tiled "
       f"--schedule static --chunks 64 --tile 64 --reps {REPS} --warmups 2 "
       f"--tdp-watts {TDP} --out {OUT}")
!{cmd}

# Limpa linhas quebradas do CSV (mantém só linhas com o nº certo de colunas)
with open(OUT, newline="") as f:
    rows = list(csv.reader(f))
ncol = len(rows[0])
good = [r for r in rows if len(r) == ncol]
with open(OUT, "w", newline="") as f:
    csv.writer(f).writerows(good)
print(f"\nCSV: {len(good)-1} resultados válidos ({len(rows)-len(good)} linha(s) quebrada(s) removida(s))")

df = pd.read_csv(OUT)
print(df["energy_source"].value_counts().to_string())
print("validação:", df["validation"].value_counts().to_dict())
if (df.energy_source == "TDP_ESTIMATE").all():
    print("ATENÇÃO: energia = TDP_ESTIMATE -> MFLOP/W é proxy de ocupação, não medida física.")
print("CV médio dos tempos: %.1f%% (acima de ~5%% = ambiente ruidoso)" % (100*df.time_cv.mean()))

files.download(OUT)


mkdir -p build
cc -std=c11 -Wall -Wextra -O3 -march=native -fopenmp -pthread src/kernels.c src/image.c src/convolution.c src/energy.c src/metrics.c src/platform.c src/main.c -o build/conv_bench -lm

--- Ambiente e fonte de energia ---
2
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
online CPUs: 2 | reps: 3 | warmups: 1 | tdp-watts: 65.0
Energy backend: none (falls back to TDP_ESTIMATE)

--- Validação do código ---
  [PASS] openmp-tiled   vs sequential  (max abs diff = 0)
=== VALIDATION PASS ===

--- Matriz completa---
== 2D Convolution Benchmark ==
online CPUs: 2 | reps: 10 | warmups: 2 | tdp-watts: 65.0
OpenMP: enabled (omp_get_num_procs=2)
CSV: /content/Artigo_PCD-main/results/results_colab.csv
Energy backend: none (falls back to TDP_ESTIMATE)
Energy note: RAPL used when present; otherwise a TDP_ESTIMATE (not a direct measurement).
CAVEATS (read before interpreting the numbers below):
  * MFLOP/W under TDP_ESTIMATE is NOT comparable across models. In
    tha

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>